# Classification Experiments

Feature engineering and classical ML experiments for 3-class beat classification (N/S/V).
Covers baseline classifiers, per-patient calibration, pre-QRS features, P-wave features, and P-wave template leakage analysis.

Builds on the core pipeline from MIT-BIH_Arrythmia.ipynb.

In [ ]:
# --- Imports and data loading ---

from ecg_monitor.pipeline import (
    build_df_all, get_train_test_split,
    FEATURE_COLS, FEATURE_COLS_28, ROBUST_RR_FEATURES,
    RR_ONLY_FEATURES, RR_ONLY_FEATURES_7, FEATURE_VARIANTS,
    DS1_RECORDS, DS2_RECORDS,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score, roc_curve, precision_recall_curve, auc,
    precision_score, recall_score,
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance

# Build full dataset
df_all, pca = build_df_all()
df_train, df_test = get_train_test_split(df_all)

y_train = df_train['clinical_label'].values
y_test = df_test['clinical_label'].values
labels_3 = ['N', 'S', 'V']

# Class distributions
print(f"\nDS1 (train): {len(df_train)} beats, DS2 (test): {len(df_test)} beats")
print(f"\n{'Class':<8} {'Train':>8} {'Test':>8}")
print(f"{'-'*8} {'-'*8} {'-'*8}")
for c in labels_3:
    n_tr = (y_train == c).sum()
    n_te = (y_test == c).sum()
    print(f"{c:<8} {n_tr:>8} {n_te:>8}")

## Baseline Classifiers (RF and GB, 34 features)

In [ ]:
# --- Single-stage classifiers (RF and GB, 34 features) ---

results_all = []  # collect results for summary table

# Random Forest
clf_rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, class_weight='balanced',
    random_state=42, n_jobs=-1,
)
clf_rf.fit(df_train[FEATURE_COLS].values, y_train)
y_pred_rf = clf_rf.predict(df_test[FEATURE_COLS].values)

# Gradient Boosting
sw = compute_sample_weight('balanced', y_train)
clf_gb = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb.fit(df_train[FEATURE_COLS].values, y_train, sample_weight=sw)
y_pred_gb = clf_gb.predict(df_test[FEATURE_COLS].values)

# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_pred, title in zip(axes, [y_pred_rf, y_pred_gb], ['Random Forest (34)', 'Gradient Boosting (34)']):
    cm = confusion_matrix(y_test, y_pred, labels=labels_3)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_3).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(title)
plt.suptitle('Single-Stage Classifiers — 34 Features', fontsize=13)
plt.tight_layout()
plt.show()

# Classification reports
for name, y_pred in [('Random Forest (34 features)', y_pred_rf),
                      ('Gradient Boosting (34 features)', y_pred_gb)]:
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, labels=labels_3, digits=3, zero_division=0))

# Store results
for name, y_pred in [('Single-stage RF (34)', y_pred_rf),
                      ('Single-stage GB (34)', y_pred_gb)]:
    results_all.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'N F1': f1_score(y_test, y_pred, labels=['N'], average=None, zero_division=0)[0],
        'S F1': f1_score(y_test, y_pred, labels=['S'], average=None, zero_division=0)[0],
        'V F1': f1_score(y_test, y_pred, labels=['V'], average=None, zero_division=0)[0],
        'Macro F1': f1_score(y_test, y_pred, average='macro', zero_division=0),
    })

## Record 232 Impact Analysis

Record 232 has 81.6% SVE burden (1,323/1,622 beats) — completely out-of-distribution
compared to DS1 max of 12.5%. It accounts for 74.8% of all DS2 S beats.

In [ ]:
# --- Record 232 impact analysis ---

test_records = df_test['record'].values
mask_232 = test_records == '232'
mask_not232 = ~mask_232

# Per-record S F1 table
print("=== Per-record S-class F1 (single-stage GB, 34 features) ===\n")
print(f"{'Record':<8} {'S beats':>8} {'S pred':>8} {'S recall':>10} {'S F1':>8}")
print("-" * 46)

for rec in sorted(df_test['record'].unique()):
    mask = test_records == rec
    true_s = (y_test[mask] == 'S').sum()
    pred_s = (y_pred_gb[mask] == 'S').sum()
    if true_s == 0 and pred_s == 0:
        continue
    s_f1 = f1_score(y_test[mask], y_pred_gb[mask], labels=['S'], average=None, zero_division=0)[0]
    tp_s = ((y_test[mask] == 'S') & (y_pred_gb[mask] == 'S')).sum()
    s_recall = tp_s / true_s if true_s > 0 else 0
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {true_s:>8} {pred_s:>8} {s_recall:>10.3f} {s_f1:>8.3f}{marker}")

# Side-by-side confusion matrices: 232 vs rest
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_232 = confusion_matrix(y_test[mask_232], y_pred_gb[mask_232], labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_232, display_labels=labels_3).plot(ax=axes[0], cmap='Oranges', values_format='d')
axes[0].set_title(f'Record 232 only ({mask_232.sum()} beats)')

cm_rest = confusion_matrix(y_test[mask_not232], y_pred_gb[mask_not232], labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_rest, display_labels=labels_3).plot(ax=axes[1], cmap='Blues', values_format='d')
axes[1].set_title(f'DS2 without 232 ({mask_not232.sum()} beats)')

plt.suptitle('Single-stage GB (34 features) — Record 232 vs Rest', fontsize=13)
plt.tight_layout()
plt.show()

# Quantify deflation
print("\n=== Record 232 impact on aggregate metrics ===\n")
for subset_name, mask in [('All DS2', np.ones(len(y_test), dtype=bool)),
                           ('Without 232', mask_not232),
                           ('232 only', mask_232)]:
    acc = accuracy_score(y_test[mask], y_pred_gb[mask])
    n_f1 = f1_score(y_test[mask], y_pred_gb[mask], labels=['N'], average=None, zero_division=0)[0]
    s_f1 = f1_score(y_test[mask], y_pred_gb[mask], labels=['S'], average=None, zero_division=0)[0]
    v_f1 = f1_score(y_test[mask], y_pred_gb[mask], labels=['V'], average=None, zero_division=0)[0]
    macro = f1_score(y_test[mask], y_pred_gb[mask], average='macro', zero_division=0)
    print(f"{subset_name:>14}: Acc={acc:.3f}  N={n_f1:.3f}  S={s_f1:.3f}  V={v_f1:.3f}  Macro={macro:.3f}")

s_f1_with = f1_score(y_test, y_pred_gb, labels=['S'], average=None, zero_division=0)[0]
s_f1_without = f1_score(y_test[mask_not232], y_pred_gb[mask_not232], labels=['S'], average=None, zero_division=0)[0]
macro_with = f1_score(y_test, y_pred_gb, average='macro', zero_division=0)
macro_without = f1_score(y_test[mask_not232], y_pred_gb[mask_not232], average='macro', zero_division=0)
print(f"\nS F1 deflation from 232: {s_f1_without - s_f1_with:+.3f}")
print(f"Macro F1 deflation from 232: {macro_without - macro_with:+.3f}")

# Store excl-232 rows for summary
results_all.append({
    'Model': 'Single-stage GB (34, excl. 232)',
    'Accuracy': accuracy_score(y_test[mask_not232], y_pred_gb[mask_not232]),
    'N F1': f1_score(y_test[mask_not232], y_pred_gb[mask_not232], labels=['N'], average=None, zero_division=0)[0],
    'S F1': s_f1_without,
    'V F1': f1_score(y_test[mask_not232], y_pred_gb[mask_not232], labels=['V'], average=None, zero_division=0)[0],
    'Macro F1': macro_without,
})

## Two-Stage Cascade Classifiers (Summary)

Two cascade architectures were tested (see Two_Stage_Classifier notebooks for full analysis):

**Cascade A** (V vs Non-V → N vs S): RF Stage 1 + GB Stage 2
- End-to-end: 91.5% accuracy, S F1 0.250, Macro F1 0.681

**Cascade B** (Normal vs Ectopic → S vs V): RF Stage 1 + GB Stage 2
- End-to-end: 94.9% accuracy, S F1 0.046, Macro F1 0.638

**Conclusion:** Neither cascade improves over single-stage GB (Macro F1 0.702). Cascades add error propagation without providing complementary information. The single-stage model already has access to all features.

In [ ]:
# --- Summary comparison table ---

df_summary = pd.DataFrame(results_all)

# Reorder: single-stage first, then cascades, then excl-232
order = ['Single-stage RF (34)', 'Single-stage GB (34)',
         'Single-stage GB (34, excl. 232)']
df_summary = df_summary.set_index('Model').loc[order].reset_index()

print("=" * 90)
print("SUMMARY: All Classification Approaches (34 features, DS1/DS2 split)")
print("=" * 90)
print()
print(df_summary.to_string(
    index=False,
    float_format='{:.3f}'.format,
    col_space={'Model': 36, 'Accuracy': 10, 'N F1': 8, 'S F1': 8, 'V F1': 8, 'Macro F1': 10}
))

# Highlight best (excluding excl-232 row for fair comparison)
df_main = df_summary[~df_summary['Model'].str.contains('excl')]
best_macro_row = df_main.loc[df_main['Macro F1'].idxmax()]
best_s_row = df_main.loc[df_main['S F1'].idxmax()]
print(f"\nBest macro F1: {best_macro_row['Model']} ({best_macro_row['Macro F1']:.3f})")
print(f"Best S F1:     {best_s_row['Model']} ({best_s_row['S F1']:.3f})")

# Styled bar chart
fig, ax = plt.subplots(figsize=(12, 5))
metrics = ['Accuracy', 'N F1', 'S F1', 'V F1', 'Macro F1']
x = np.arange(len(metrics))
width = 0.18
models_to_plot = df_main['Model'].values

for i, model in enumerate(models_to_plot):
    row = df_main[df_main['Model'] == model].iloc[0]
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=model)

ax.set_xticks(x + width * (len(models_to_plot) - 1) / 2)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('All Approaches Compared (34 features)')
ax.legend(loc='lower left', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# --- Feature importance ---

# GB permutation importance (macro F1 scorer)
from sklearn.metrics import make_scorer

def macro_f1_scorer(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

scorer = make_scorer(macro_f1_scorer)
perm_imp = permutation_importance(
    clf_gb, df_test[FEATURE_COLS].values, y_test,
    scoring=scorer, n_repeats=10, random_state=42, n_jobs=-1,
)

# RF Gini importance
rf_importances = clf_rf.feature_importances_

# Top 15 for each
perm_idx = np.argsort(perm_imp.importances_mean)[::-1][:15]
gini_idx = np.argsort(rf_importances)[::-1][:15]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# GB permutation importance
axes[0].barh(range(15), perm_imp.importances_mean[perm_idx][::-1], color='steelblue')
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([FEATURE_COLS[i] for i in perm_idx][::-1])
axes[0].set_xlabel('Mean decrease in macro F1')
axes[0].set_title('GB Permutation Importance (top 15)')
axes[0].grid(axis='x', alpha=0.3)

# RF Gini importance
axes[1].barh(range(15), rf_importances[gini_idx][::-1], color='coral')
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([FEATURE_COLS[i] for i in gini_idx][::-1])
axes[1].set_xlabel('Gini importance')
axes[1].set_title('RF Gini Importance (top 15)')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance — 34-Feature Set', fontsize=13)
plt.tight_layout()
plt.show()

# Print tables
print("=== GB Permutation Importance (top 15, by macro F1) ===\n")
print(f"{'Feature':<30} {'Importance':>12} {'Std':>10}")
print("-" * 54)
for i in perm_idx:
    print(f"{FEATURE_COLS[i]:<30} {perm_imp.importances_mean[i]:>12.4f} {perm_imp.importances_std[i]:>10.4f}")

print(f"\n=== RF Gini Importance (top 15) ===\n")
print(f"{'Feature':<30} {'Importance':>12}")
print("-" * 44)
for i in gini_idx:
    print(f"{FEATURE_COLS[i]:<30} {rf_importances[i]:>12.4f}")

# Key Findings (34-Feature Baseline)

## Best approach: Single-stage Gradient Boosting (34 features)
- **Macro F1 ~0.702**, outperforming all cascade variants and single-stage RF
- 34-feature set (28 original + 6 robust RR) provides consistent improvement over 28-feature baseline (macro F1 0.691 → 0.702)

## Cascades don't help
- Both cascade orderings (V-first and ectopic-first) add error propagation without complementary information
- Single-stage model with all 34 features already captures the same signal

## Record 232: out-of-distribution effect
- Record 232 has 81.6% SVE burden — DS1 max is only 12.5%
- Removing 232: S F1 improves by ~+0.12, macro F1 by ~+0.04
- Root cause: per-patient normalized features and local-window RR ratios are corrupted when S beats dominate

## Remaining bottleneck: S-class detection
- S F1 ~0.23 is limited by data and features, not model capacity
- Path forward: P-wave morphology features, per-patient calibration, cross-lead validation

# Per-Patient Calibration Simulation

**Simulate deployment calibration:** For each patient, take the first 5 minutes of ground-truth N-labeled beats as a "calibration window." Compute that patient's N-beat baseline (mean/std) from this window only, then normalize all their beats against it.

**Why this differs from existing per-patient norms:** Current `_norm` features use ALL beats from a patient — when S beats dominate (record 232), the baseline IS the abnormal rhythm. Calibrated norms use only confirmed-N beats from a short initial window, so the baseline reflects the patient's actual healthy conduction.

**Deployment analogue:** Clinician attaches device, patient rests for 5 minutes, system establishes baseline. All subsequent monitoring is relative to that baseline.

In [ ]:
# --- Per-patient calibration from first 5 min of ground-truth N beats ---

CALIB_FEATURES = ['r_amplitude', 'qrs_width_ms', 'qrs_area', 'qrs_range',
                  'rr_prev', 'rr_curr']
CALIB_DURATION_SAMPLES = 5 * 60 * 360  # 5 minutes at 360 Hz

def compute_calibrated_norms(df, calib_features, calib_duration=CALIB_DURATION_SAMPLES):
    """For each record, compute N-beat baseline from first 5 min of ground-truth N beats,
    then normalize all beats in that record against the calibrated baseline."""
    df = df.copy()
    calib_cols = [f'{f}_calib' for f in calib_features]

    # Initialize with NaN
    for col in calib_cols:
        df[col] = np.nan

    calib_stats = {}
    for rec in df['record'].unique():
        rec_mask = df['record'] == rec
        rec_df = df[rec_mask]

        # First 5 min of N beats only
        n_beats = rec_df[(rec_df['clinical_label'] == 'N') &
                         (rec_df['sample_idx'] <= calib_duration)]

        if len(n_beats) < 10:
            # Fallback: use all N beats from first 5 min regardless, or expand window
            n_beats = rec_df[rec_df['clinical_label'] == 'N'].head(30)

        stats = {}
        for feat in calib_features:
            mean_val = n_beats[feat].mean()
            std_val = n_beats[feat].std()
            if std_val == 0 or np.isnan(std_val):
                std_val = 1.0
            stats[feat] = {'mean': mean_val, 'std': std_val}

        calib_stats[rec] = stats

        # Normalize all beats in this record
        for feat in calib_features:
            col = f'{feat}_calib'
            df.loc[rec_mask, col] = (
                (df.loc[rec_mask, feat] - stats[feat]['mean']) / stats[feat]['std']
            )

    return df, calib_stats

# Apply calibration to both train and test
df_train_calib, calib_stats_train = compute_calibrated_norms(df_train, CALIB_FEATURES)
df_test_calib, calib_stats_test = compute_calibrated_norms(df_test, CALIB_FEATURES)

calib_cols = [f'{f}_calib' for f in CALIB_FEATURES]
print(f"Created {len(calib_cols)} calibrated features: {calib_cols}\n")

# Show calibration window stats for key records
print("=== Calibration window stats (selected DS2 records) ===\n")
print(f"{'Record':<8} {'N beats (5min)':>14} {'SVE burden':>12} {'r_amp mean':>12} {'rr_prev mean':>12}")
print("-" * 62)
for rec in ['100', '213', '222', '232']:
    if rec in calib_stats_test:
        stats = calib_stats_test[rec]
        rec_df = df_test[df_test['record'] == rec]
        n_5min = len(rec_df[(rec_df['clinical_label'] == 'N') &
                            (rec_df['sample_idx'] <= CALIB_DURATION_SAMPLES)])
        sve_pct = (rec_df['clinical_label'] == 'S').sum() / len(rec_df) * 100
        print(f"{rec:<8} {n_5min:>14} {sve_pct:>11.1f}% {stats['r_amplitude']['mean']:>12.3f} {stats['rr_prev']['mean']:>12.3f}")

# Show how calibrated features separate classes on record 232
print(f"\n=== Record 232: Calibrated feature distributions by class ===\n")
rec232 = df_test_calib[df_test_calib['record'] == '232']
print(f"{'Feature':<25} {'N mean':>8} {'S mean':>8} {'Separation':>12}")
print("-" * 56)
for col in calib_cols:
    n_mean = rec232.loc[rec232['clinical_label'] == 'N', col].mean()
    s_mean = rec232.loc[rec232['clinical_label'] == 'S', col].mean()
    sep = abs(s_mean - n_mean)
    print(f"{col:<25} {n_mean:>8.3f} {s_mean:>8.3f} {sep:>12.3f}")

# Compare with the existing per-patient norm on record 232
print(f"\n=== Record 232: Per-patient norm (corrupted) vs Calibrated norm ===\n")
existing_norms = ['r_amplitude_norm', 'qrs_width_ms_norm', 'qrs_area_norm']
print(f"{'Feature':<25} {'N mean':>8} {'S mean':>8} {'|Sep|':>8}")
print("-" * 52)
for col in existing_norms:
    n_mean = rec232.loc[rec232['clinical_label'] == 'N', col].mean()
    s_mean = rec232.loc[rec232['clinical_label'] == 'S', col].mean()
    print(f"{col:<25} {n_mean:>8.3f} {s_mean:>8.3f} {abs(s_mean - n_mean):>8.3f}")
print("  --- vs calibrated ---")
for col in [f'{f}_calib' for f in ['r_amplitude', 'qrs_width_ms', 'qrs_area']]:
    n_mean = rec232.loc[rec232['clinical_label'] == 'N', col].mean()
    s_mean = rec232.loc[rec232['clinical_label'] == 'S', col].mean()
    print(f"{col:<25} {n_mean:>8.3f} {s_mean:>8.3f} {abs(s_mean - n_mean):>8.3f}")

In [ ]:
# --- Train GB with calibrated features and compare ---

# Strategy C: Replace per-patient norms with calibrated norms
FEATURES_CALIB_REPLACE = [f for f in FEATURE_COLS
                          if f not in ['r_amplitude_norm', 'qrs_width_ms_norm', 'qrs_area_norm']
                          ] + calib_cols
print(f"Strategy C (replace per-patient with calibrated): {len(FEATURES_CALIB_REPLACE)} features")

# Strategy D: Keep per-patient norms AND add calibrated norms
FEATURES_CALIB_BOTH = FEATURE_COLS + calib_cols
print(f"Strategy D (keep both + calibrated): {len(FEATURES_CALIB_BOTH)} features\n")

calib_strategies = {
    'GB calib replace (37)': FEATURES_CALIB_REPLACE,
    'GB calib + per-patient (40)': FEATURES_CALIB_BOTH,
}

calib_results = {}
for name, feat_cols in calib_strategies.items():
    sw_c = compute_sample_weight('balanced', y_train)
    clf_c = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.1,
        min_samples_leaf=20, l2_regularization=1.0, random_state=42,
    )
    clf_c.fit(df_train_calib[feat_cols].values, y_train, sample_weight=sw_c)
    y_pred_c = clf_c.predict(df_test_calib[feat_cols].values)
    calib_results[name] = {'y_pred': y_pred_c, 'clf': clf_c, 'feat_cols': feat_cols}

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred_c, labels=labels_3, digits=3, zero_division=0))

    results_all.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred_c),
        'N F1': f1_score(y_test, y_pred_c, labels=['N'], average=None, zero_division=0)[0],
        'S F1': f1_score(y_test, y_pred_c, labels=['S'], average=None, zero_division=0)[0],
        'V F1': f1_score(y_test, y_pred_c, labels=['V'], average=None, zero_division=0)[0],
        'Macro F1': f1_score(y_test, y_pred_c, average='macro', zero_division=0),
    })

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, res) in zip(axes, calib_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'], labels=labels_3)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_3).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(name)
plt.suptitle('Per-Patient Calibrated Normalization — Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Record 232 deep dive — calibrated vs baseline ---

# Pick best calibrated strategy
best_calib_name = max(calib_results.keys(),
                      key=lambda k: f1_score(y_test, calib_results[k]['y_pred'], average='macro', zero_division=0))
best_calib_pred = calib_results[best_calib_name]['y_pred']

print(f"Best calibrated strategy: {best_calib_name}\n")

# Overall and record-232 comparison
print("=== Calibrated vs Baseline: Subset Comparison ===\n")
print(f"{'Subset':<16} {'Metric':<10} {'Baseline GB (34)':>18} {best_calib_name:>28} {'Delta':>8}")
print("-" * 84)

for subset_name, mask in [('All DS2', np.ones(len(y_test), dtype=bool)),
                           ('Without 232', mask_not232),
                           ('232 only', mask_232)]:
    for metric_name, metric_fn in [('S F1', lambda yt, yp: f1_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('V F1', lambda yt, yp: f1_score(yt, yp, labels=['V'], average=None, zero_division=0)[0]),
                                    ('Macro F1', lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0))]:
        base_val = metric_fn(y_test[mask], y_pred_gb[mask])
        calib_val = metric_fn(y_test[mask], best_calib_pred[mask])
        delta = calib_val - base_val
        print(f"{subset_name:<16} {metric_name:<10} {base_val:>18.3f} {calib_val:>28.3f} {delta:>+8.3f}")
    print()

# Per-record S F1
print(f"\n=== Per-record S F1: Baseline vs Calibrated ===\n")
print(f"{'Record':<8} {'S beats':>8} {'Base S F1':>10} {'Calib S F1':>11} {'Delta':>8}")
print("-" * 49)

for rec in sorted(df_test['record'].unique()):
    mask_rec = test_records == rec
    true_s = (y_test[mask_rec] == 'S').sum()
    if true_s == 0:
        continue
    s_f1_base = f1_score(y_test[mask_rec], y_pred_gb[mask_rec], labels=['S'], average=None, zero_division=0)[0]
    s_f1_calib = f1_score(y_test[mask_rec], best_calib_pred[mask_rec], labels=['S'], average=None, zero_division=0)[0]
    delta = s_f1_calib - s_f1_base
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {true_s:>8} {s_f1_base:>10.3f} {s_f1_calib:>11.3f} {delta:>+8.3f}{marker}")

# Side-by-side confusion matrices for record 232
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_base_232 = confusion_matrix(y_test[mask_232], y_pred_gb[mask_232], labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_base_232, display_labels=labels_3).plot(ax=axes[0], cmap='Oranges', values_format='d')
axes[0].set_title('Record 232 — Baseline GB (34)')

cm_calib_232 = confusion_matrix(y_test[mask_232], best_calib_pred[mask_232], labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_calib_232, display_labels=labels_3).plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title(f'Record 232 — {best_calib_name}')

plt.suptitle('Record 232: Baseline vs Calibrated Normalization', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Final summary table with all approaches ---

df_summary_final = pd.DataFrame(results_all)

order_final = [
    'Single-stage RF (34)', 'Single-stage GB (34)',
    'GB calib replace (37)', 'GB calib + per-patient (40)',
    'Single-stage GB (34, excl. 232)',
]
order_final = [m for m in order_final if m in df_summary_final['Model'].values]
df_summary_final = df_summary_final.set_index('Model').loc[order_final].reset_index()

print("=" * 95)
print("SUMMARY: Baseline + Calibration Approaches")
print("=" * 95)
print()
print(df_summary_final.to_string(
    index=False,
    float_format='{:.3f}'.format,
    col_space={'Model': 40, 'Accuracy': 10, 'N F1': 8, 'S F1': 8, 'V F1': 8, 'Macro F1': 10}
))

df_main_final = df_summary_final[~df_summary_final['Model'].str.contains('excl')]
best_macro = df_main_final.loc[df_main_final['Macro F1'].idxmax()]
best_s = df_main_final.loc[df_main_final['S F1'].idxmax()]
print(f"\nBest macro F1: {best_macro['Model']} ({best_macro['Macro F1']:.3f})")
print(f"Best S F1:     {best_s['Model']} ({best_s['S F1']:.3f})")

# Bar chart — just the key approaches
key_models = ['Single-stage GB (34)', 'GB calib replace (37)', 'GB calib + per-patient (40)']
key_models = [m for m in key_models if m in df_summary_final['Model'].values]
df_key = df_summary_final[df_summary_final['Model'].isin(key_models)]

fig, ax = plt.subplots(figsize=(12, 5))
metrics = ['Accuracy', 'N F1', 'S F1', 'V F1', 'Macro F1']
x = np.arange(len(metrics))
width = 0.25

for i, model in enumerate(key_models):
    row = df_key[df_key['Model'] == model].iloc[0]
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=model)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Baseline vs Calibrated Normalization')
ax.legend(loc='lower left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Pre-QRS Features v2: Fixed Windows + Per-Patient Normalization

**Changes from v1:**
1. **Fixed ms windows** anchored to R-peaks instead of percentage-based splits
2. **Per-patient normalization** of all pre-QRS features against each patient's N-beat baseline
3. **Template correlation** — correlation of preceding T-wave against patient's average N-beat T-wave
4. **Keep all features** — re-evaluate Cohen's d after normalization before deciding what to drop

In [ ]:
# --- Extract pre-QRS v2 features with fixed ms windows ---

def extract_preqrs_features_v2(data_dir, rec_id, df_rec, fs=360):
    """Extract inter-beat window features using fixed physiological ms offsets.
    
    T-wave window:  R[i-1]+100ms to R[i-1]+400ms (preceding beat's T-wave)
    Pre-QRS window: R[i]-250ms to R[i]-50ms (current beat's P-wave region)
    """
    record = wfdb.rdrecord(f'{data_dir}/{rec_id}')
    _, filtered = pan_tompkins_detect(record.p_signal[:, 0], record.fs)
    
    # Fixed offsets in samples
    tw_start_offset = int(0.100 * fs)   # 100ms after preceding R
    tw_end_offset = int(0.400 * fs)     # 400ms after preceding R
    pq_start_offset = int(0.250 * fs)   # 250ms before current R
    pq_end_offset = int(0.050 * fs)     # 50ms before current R
    
    nan_cols = ['tw_amplitude', 'tw_energy', 'tw_max', 'tw_min', 'tw_range',
                'tw_skew', 'tw_kurt', 'tw_slope_late', 'tw_residual_energy',
                'preqrs_amplitude', 'preqrs_slope', 'preqrs_energy',
                'tw_waveform']
    
    features = []
    sample_indices = df_rec['sample_idx'].values
    
    for idx, row in df_rec.iterrows():
        r_curr = int(row['sample_idx'])
        pos = np.searchsorted(sample_indices, r_curr)
        
        if pos <= 0 or pos >= len(sample_indices):
            feat = {col: np.nan for col in nan_cols if col != 'tw_waveform'}
            feat['tw_waveform'] = None
            feat['sample_idx'] = r_curr
            features.append(feat)
            continue
        
        r_prev = int(sample_indices[pos - 1])
        
        # T-wave window: R[i-1]+100ms to R[i-1]+400ms
        tw_start = r_prev + tw_start_offset
        tw_end = min(r_prev + tw_end_offset, r_curr - pq_end_offset)  # don't overlap into current QRS
        
        # Pre-QRS window: R[i]-250ms to R[i]-50ms
        pq_start = max(r_curr - pq_start_offset, r_prev + tw_start_offset)  # don't go before preceding QRS
        pq_end = r_curr - pq_end_offset
        
        # Validate windows
        if (tw_end <= tw_start or tw_start < 0 or tw_end >= len(filtered) or
            pq_end <= pq_start or pq_start < 0 or pq_end >= len(filtered)):
            feat = {col: np.nan for col in nan_cols if col != 'tw_waveform'}
            feat['tw_waveform'] = None
            feat['sample_idx'] = r_curr
            features.append(feat)
            continue
        
        tw_segment = filtered[tw_start:tw_end]
        pq_segment = filtered[pq_start:pq_end]
        
        if len(tw_segment) < 5 or len(pq_segment) < 5:
            feat = {col: np.nan for col in nan_cols if col != 'tw_waveform'}
            feat['tw_waveform'] = None
            feat['sample_idx'] = r_curr
            features.append(feat)
            continue
        
        kern_size = max(1, len(tw_segment) // 4)
        
        feat = {
            'sample_idx': r_curr,
            # T-wave features
            'tw_amplitude': np.max(tw_segment) - tw_segment[0],
            'tw_energy': np.sum(tw_segment ** 2) / len(tw_segment),
            'tw_max': np.max(tw_segment),
            'tw_min': np.min(tw_segment),
            'tw_range': np.max(tw_segment) - np.min(tw_segment),
            'tw_skew': skew(tw_segment),
            'tw_kurt': kurtosis(tw_segment),
            'tw_slope_late': np.mean(np.diff(tw_segment[len(tw_segment)//2:])),
            'tw_residual_energy': np.sum((tw_segment - np.convolve(tw_segment,
                                         np.ones(kern_size) / kern_size,
                                         mode='same')) ** 2) / len(tw_segment),
            # Pre-QRS features
            'preqrs_amplitude': np.max(np.abs(pq_segment)),
            'preqrs_slope': np.mean(np.diff(pq_segment)),
            'preqrs_energy': np.sum(pq_segment ** 2) / len(pq_segment),
            # Store raw T-wave waveform for template correlation (computed later per-patient)
            'tw_waveform': tw_segment.copy(),
        }
        features.append(feat)
    
    return pd.DataFrame(features)

# Extract v2 features for all records
preqrs_v2_dfs = []
for rec_id in sorted(df_all['record'].unique()):
    df_rec = df_all[df_all['record'] == rec_id]
    pq_feats = extract_preqrs_features_v2(DATA_DIR, rec_id, df_rec)
    preqrs_v2_dfs.append(pq_feats)
    n_valid = pq_feats['tw_amplitude'].notna().sum()
    print(f"Record {rec_id}: {len(pq_feats)} beats, {len(pq_feats) - n_valid} NaN rows")

df_preqrs_v2 = pd.concat(preqrs_v2_dfs, ignore_index=True)

PREQRS_V2_COLS = ['tw_amplitude', 'tw_energy', 'tw_max', 'tw_min', 'tw_range',
                  'tw_skew', 'tw_kurt', 'tw_slope_late', 'tw_residual_energy',
                  'preqrs_amplitude', 'preqrs_slope', 'preqrs_energy']

print(f"\nExtracted {len(PREQRS_V2_COLS)} pre-QRS v2 features for {len(df_preqrs_v2)} beats")
print(f"NaN beats: {df_preqrs_v2[PREQRS_V2_COLS].isna().any(axis=1).sum()}")

In [ ]:
# --- Per-patient normalize + template correlation ---

from scipy.signal import resample as sig_resample

# Merge v2 features into df_all
df_all_v2 = df_all.copy().reset_index(drop=True)
df_preqrs_v2_aligned = df_preqrs_v2.reset_index(drop=True)

for col in PREQRS_V2_COLS:
    df_all_v2[col] = df_preqrs_v2_aligned[col].values
df_all_v2['tw_waveform_v2'] = df_preqrs_v2_aligned['tw_waveform'].values

# Drop NaN rows
n_before = len(df_all_v2)
valid_mask = df_all_v2[PREQRS_V2_COLS].notna().all(axis=1)
df_all_v2 = df_all_v2[valid_mask].copy()
print(f"Dropped {n_before - len(df_all_v2)} NaN rows, {len(df_all_v2)} beats remaining")

# --- Template correlation ---
# For each record: compute average T-wave waveform from N beats, then correlate each beat's T-wave
TEMPLATE_LEN = 72  # resample all T-wave waveforms to fixed length (200ms at 360 Hz ≈ 72 samples)

def compute_template_correlation(df, template_len=TEMPLATE_LEN):
    """Compute T-wave template from N beats per record, then correlate each beat."""
    df = df.copy()
    df['tw_template_corr'] = np.nan
    
    for rec in df['record'].unique():
        rec_mask = df['record'] == rec
        rec_df = df[rec_mask]
        
        # Get N-beat T-wave waveforms for template
        n_waveforms = []
        for wf in rec_df.loc[rec_df['clinical_label'] == 'N', 'tw_waveform_v2']:
            if wf is not None and len(wf) >= 5:
                resampled = sig_resample(wf, template_len)
                n_waveforms.append(resampled)
        
        if len(n_waveforms) < 5:
            continue
        
        # Average N-beat T-wave template
        template = np.mean(n_waveforms, axis=0)
        template_norm = template - np.mean(template)
        template_std = np.std(template_norm)
        if template_std == 0:
            continue
        
        # Correlate each beat's T-wave against the template
        corr_values = []
        indices = []
        for idx, wf in zip(rec_df.index, rec_df['tw_waveform_v2']):
            if wf is not None and len(wf) >= 5:
                resampled = sig_resample(wf, template_len)
                beat_norm = resampled - np.mean(resampled)
                beat_std = np.std(beat_norm)
                if beat_std > 0:
                    corr = np.corrcoef(template_norm, beat_norm)[0, 1]
                else:
                    corr = 0.0
                corr_values.append(corr)
                indices.append(idx)
        
        df.loc[indices, 'tw_template_corr'] = corr_values
    
    return df

df_all_v2 = compute_template_correlation(df_all_v2)
print(f"Template correlation NaNs: {df_all_v2['tw_template_corr'].isna().sum()}")

# Drop any remaining NaNs from template correlation
df_all_v2 = df_all_v2.dropna(subset=['tw_template_corr']).copy()
print(f"Final dataset: {len(df_all_v2)} beats")

# --- Per-patient normalization of pre-QRS features ---
for feat in PREQRS_V2_COLS:
    group_mean = df_all_v2.groupby('record')[feat].transform('mean')
    group_std = df_all_v2.groupby('record')[feat].transform('std').replace(0, 1)
    df_all_v2[f'{feat}_pnorm'] = (df_all_v2[feat] - group_mean) / group_std

PREQRS_V2_PNORM_COLS = [f'{feat}_pnorm' for feat in PREQRS_V2_COLS]
print(f"\nCreated {len(PREQRS_V2_PNORM_COLS)} per-patient normalized features + tw_template_corr")

# --- Split ---
train_mask_v2 = df_all_v2['record'].isin(DS1_RECORDS)
test_mask_v2 = df_all_v2['record'].isin(DS2_RECORDS)
df_train_v2 = df_all_v2[train_mask_v2].copy()
df_test_v2 = df_all_v2[test_mask_v2].copy()
y_train_v2 = df_train_v2['clinical_label'].values
y_test_v2 = df_test_v2['clinical_label'].values
print(f"Train: {len(df_train_v2)}, Test: {len(df_test_v2)}")

In [ ]:
# --- Re-evaluate Cohen's d after per-patient normalization ---

ALL_PREQRS_V2 = PREQRS_V2_COLS + PREQRS_V2_PNORM_COLS + ['tw_template_corr']

print("=== Cohen's d for N-vs-S separation: Raw vs Per-patient Normalized ===\n")
print(f"{'Feature':<30} {'Raw d':>8} {'PNorm d':>8} {'Change':>8}")
print("-" * 58)

cohens_d_results = {}
for feat in PREQRS_V2_COLS:
    # Raw
    n_vals = df_test_v2.loc[df_test_v2['clinical_label'] == 'N', feat]
    s_vals = df_test_v2.loc[df_test_v2['clinical_label'] == 'S', feat]
    pooled = np.sqrt((n_vals.std()**2 + s_vals.std()**2) / 2)
    raw_d = abs(n_vals.mean() - s_vals.mean()) / pooled if pooled > 0 else 0

    # Per-patient normalized
    feat_pn = f'{feat}_pnorm'
    n_vals_pn = df_test_v2.loc[df_test_v2['clinical_label'] == 'N', feat_pn]
    s_vals_pn = df_test_v2.loc[df_test_v2['clinical_label'] == 'S', feat_pn]
    pooled_pn = np.sqrt((n_vals_pn.std()**2 + s_vals_pn.std()**2) / 2)
    pnorm_d = abs(n_vals_pn.mean() - s_vals_pn.mean()) / pooled_pn if pooled_pn > 0 else 0

    change = pnorm_d - raw_d
    cohens_d_results[feat] = {'raw': raw_d, 'pnorm': pnorm_d}
    marker = " ***" if pnorm_d > raw_d + 0.1 else ""
    print(f"{feat:<30} {raw_d:>8.3f} {pnorm_d:>8.3f} {change:>+8.3f}{marker}")

# Template correlation
n_corr = df_test_v2.loc[df_test_v2['clinical_label'] == 'N', 'tw_template_corr']
s_corr = df_test_v2.loc[df_test_v2['clinical_label'] == 'S', 'tw_template_corr']
v_corr = df_test_v2.loc[df_test_v2['clinical_label'] == 'V', 'tw_template_corr']
pooled_corr = np.sqrt((n_corr.std()**2 + s_corr.std()**2) / 2)
corr_d = abs(n_corr.mean() - s_corr.mean()) / pooled_corr if pooled_corr > 0 else 0
print(f"\n{'tw_template_corr':<30} {'—':>8} {corr_d:>8.3f} {'(new)':>8}")
print(f"  N mean={n_corr.mean():.3f}, S mean={s_corr.mean():.3f}, V mean={v_corr.mean():.3f}")

# Visualize template correlation distribution
fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [('N', 'steelblue'), ('S', 'orange'), ('V', 'red')]:
    data = df_test_v2.loc[df_test_v2['clinical_label'] == label, 'tw_template_corr']
    ax.hist(data, bins=60, alpha=0.5, label=f'{label} (n={len(data)})', color=color, density=True)
ax.set_xlabel('T-wave Template Correlation')
ax.set_ylabel('Density')
ax.set_title('T-wave Template Correlation by Class (DS2)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Train GB with v2 pre-QRS features and compare ---

# Feature set: 34 original + 12 raw pre-QRS + 12 per-patient normalized pre-QRS + 1 template corr = 59
FEATURES_V2_ALL = FEATURE_COLS + PREQRS_V2_COLS + PREQRS_V2_PNORM_COLS + ['tw_template_corr']
print(f"Full v2 feature set: {len(FEATURES_V2_ALL)} features\n")

# Train GB with all v2 features
sw_v2 = compute_sample_weight('balanced', y_train_v2)
clf_gb_v2 = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_v2.fit(df_train_v2[FEATURES_V2_ALL].values, y_train_v2, sample_weight=sw_v2)
y_pred_v2 = clf_gb_v2.predict(df_test_v2[FEATURES_V2_ALL].values)

# Baseline on matched dataset
clf_gb_base_v2 = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_base_v2.fit(df_train_v2[FEATURE_COLS].values, y_train_v2, sample_weight=sw_v2)
y_pred_base_v2 = clf_gb_base_v2.predict(df_test_v2[FEATURE_COLS].values)

print("=== Baseline GB (34 features, matched dataset) ===")
print(classification_report(y_test_v2, y_pred_base_v2, labels=labels_3, digits=3, zero_division=0))

print("=== GB + Pre-QRS v2 (59 features) ===")
print(classification_report(y_test_v2, y_pred_v2, labels=labels_3, digits=3, zero_division=0))

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_base = confusion_matrix(y_test_v2, y_pred_base_v2, labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_base, display_labels=labels_3).plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Baseline GB (34)')

cm_v2 = confusion_matrix(y_test_v2, y_pred_v2, labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_v2, display_labels=labels_3).plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title('GB + Pre-QRS v2 (59)')

plt.suptitle('Pre-QRS v2: Fixed Windows + Per-Patient Norm + Template Corr', fontsize=13)
plt.tight_layout()
plt.show()

# Subset analysis
test_records_v2 = df_test_v2['record'].values
mask_232_v2 = test_records_v2 == '232'
mask_not232_v2 = ~mask_232_v2

print("\n=== Subset Comparison ===\n")
print(f"{'Subset':<16} {'Metric':<10} {'Baseline (34)':>14} {'+ PreQRS v2 (59)':>18} {'Delta':>8}")
print("-" * 70)

for subset_name, mask in [('All DS2', np.ones(len(y_test_v2), dtype=bool)),
                           ('Without 232', mask_not232_v2),
                           ('232 only', mask_232_v2)]:
    for metric_name, metric_fn in [('S F1', lambda yt, yp: f1_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('V F1', lambda yt, yp: f1_score(yt, yp, labels=['V'], average=None, zero_division=0)[0]),
                                    ('Macro F1', lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0))]:
        base_val = metric_fn(y_test_v2[mask], y_pred_base_v2[mask])
        v2_val = metric_fn(y_test_v2[mask], y_pred_v2[mask])
        delta = v2_val - base_val
        print(f"{subset_name:<16} {metric_name:<10} {base_val:>14.3f} {v2_val:>18.3f} {delta:>+8.3f}")
    print()

# Per-record S F1
print("=== Per-record S F1: Baseline vs + Pre-QRS v2 ===\n")
print(f"{'Record':<8} {'S beats':>8} {'Base S F1':>10} {'v2 S F1':>10} {'Delta':>8}")
print("-" * 48)
for rec in sorted(df_test_v2['record'].unique()):
    mask_rec = test_records_v2 == rec
    true_s = (y_test_v2[mask_rec] == 'S').sum()
    if true_s == 0:
        continue
    s_base = f1_score(y_test_v2[mask_rec], y_pred_base_v2[mask_rec], labels=['S'], average=None, zero_division=0)[0]
    s_v2 = f1_score(y_test_v2[mask_rec], y_pred_v2[mask_rec], labels=['S'], average=None, zero_division=0)[0]
    delta = s_v2 - s_base
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {true_s:>8} {s_base:>10.3f} {s_v2:>10.3f} {delta:>+8.3f}{marker}")

# P-Wave Presence Detection from MIT-BIH

**Approach:** Use the beat labels as implicit P-wave ground truth:
- N beats → P-wave present (normal sinus rhythm)
- V beats → P-wave absent (ventricular origin)
- S beats → P-wave absent/abnormal (what we want to detect)

**P-wave window:** ~200ms to ~80ms before R-peak. On MLII, normal P-waves appear as a small upward deflection in this region. At 360 Hz this is 72 to 29 samples before R-peak — a narrow window focused on where the P-wave actually lives.

**Method:** Per-patient template matching — average the P-wave window across N beats to build a template, then correlate each beat's P-wave window against it. N beats should match; S and V beats should not.

In [ ]:
# --- Extract P-wave window features ---

from scipy.signal import find_peaks

def extract_pwave_features(data_dir, rec_id, df_rec, fs=360):
    """Extract features from the P-wave window (~200ms to ~80ms before R-peak).
    
    Returns per-beat features + raw P-wave waveform for template matching.
    """
    record = wfdb.rdrecord(f'{data_dir}/{rec_id}')
    _, filtered = pan_tompkins_detect(record.p_signal[:, 0], record.fs)
    
    # P-wave window: 200ms to 80ms before R-peak
    pw_start_offset = int(0.200 * fs)  # 72 samples before R
    pw_end_offset = int(0.080 * fs)    # 29 samples before R
    pw_len = pw_start_offset - pw_end_offset  # 43 samples (~120ms)
    
    sample_indices = df_rec['sample_idx'].values
    features = []
    
    nan_row = {
        'pw_peak_amplitude': np.nan, 'pw_energy': np.nan,
        'pw_peak_prominence': np.nan, 'pw_has_peak': np.nan,
        'pw_baseline_dev': np.nan, 'pw_max': np.nan,
        'pw_min': np.nan, 'pw_range': np.nan, 'pw_area': np.nan,
    }
    
    for idx, row in df_rec.iterrows():
        r_curr = int(row['sample_idx'])
        
        pw_start = r_curr - pw_start_offset
        pw_end = r_curr - pw_end_offset
        
        # Check preceding R-peak — don't let P-wave window overlap into previous QRS
        pos = np.searchsorted(sample_indices, r_curr)
        if pos > 0:
            r_prev = int(sample_indices[pos - 1])
            prev_qrs_end = r_prev + int(0.100 * fs)
            if pw_start < prev_qrs_end:
                pw_start = prev_qrs_end
        
        if pw_start < 0 or pw_end >= len(filtered) or pw_end <= pw_start or (pw_end - pw_start) < 5:
            feat = nan_row.copy()
            feat['sample_idx'] = r_curr
            feat['pw_waveform'] = None
            features.append(feat)
            continue
        
        pw_segment = filtered[pw_start:pw_end]
        
        # Baseline: average of first and last 3 samples of the window
        baseline = (np.mean(pw_segment[:3]) + np.mean(pw_segment[-3:])) / 2
        pw_detrended = pw_segment - baseline
        
        # Peak detection in P-wave window
        peaks, properties = find_peaks(pw_detrended, prominence=0.01)
        has_peak = len(peaks) > 0
        if has_peak:
            best_peak = peaks[np.argmax(properties['prominences'])]
            peak_amp = pw_detrended[best_peak]
            peak_prom = properties['prominences'][np.argmax(properties['prominences'])]
        else:
            peak_amp = np.max(pw_detrended)
            peak_prom = 0.0
        
        # Resample to fixed length for template matching
        if len(pw_segment) >= 5:
            pw_resampled = sig_resample(pw_segment, pw_len)
        else:
            pw_resampled = None
        
        features.append({
            'sample_idx': r_curr,
            'pw_waveform': pw_resampled,
            'pw_peak_amplitude': peak_amp,
            'pw_energy': np.sum(pw_detrended ** 2) / len(pw_detrended),
            'pw_peak_prominence': peak_prom,
            'pw_has_peak': float(has_peak),
            'pw_baseline_dev': np.mean(np.abs(pw_detrended)),
            'pw_max': np.max(pw_segment),
            'pw_min': np.min(pw_segment),
            'pw_range': np.max(pw_segment) - np.min(pw_segment),
            'pw_area': np.trapezoid(pw_detrended) / fs,
        })
    
    return pd.DataFrame(features)

# Extract for all records
pwave_dfs = []
for rec_id in sorted(df_all['record'].unique()):
    df_rec = df_all[df_all['record'] == rec_id]
    pw_feats = extract_pwave_features(DATA_DIR, rec_id, df_rec)
    pwave_dfs.append(pw_feats)
    n_valid = pw_feats['pw_energy'].notna().sum()
    print(f"Record {rec_id}: {len(pw_feats)} beats, {len(pw_feats) - n_valid} invalid")

df_pwave = pd.concat(pwave_dfs, ignore_index=True)

PW_FEAT_COLS = ['pw_peak_amplitude', 'pw_energy', 'pw_peak_prominence',
                'pw_has_peak', 'pw_baseline_dev', 'pw_max', 'pw_min',
                'pw_range', 'pw_area']

print(f"\nExtracted {len(PW_FEAT_COLS)} P-wave features for {len(df_pwave)} beats")
print(f"Invalid beats: {df_pwave[PW_FEAT_COLS].isna().any(axis=1).sum()}")

In [ ]:
# --- P-wave template matching + per-patient normalization ---

PW_TEMPLATE_LEN = int(0.200 * 360) - int(0.080 * 360)  # 44 samples (~120ms at 360 Hz)

# Merge P-wave features into df_all
df_all_pw = df_all.copy().reset_index(drop=True)
df_pwave_aligned = df_pwave.reset_index(drop=True)

for col in PW_FEAT_COLS:
    df_all_pw[col] = df_pwave_aligned[col].values
df_all_pw['pw_waveform'] = df_pwave_aligned['pw_waveform'].values

# Drop NaN rows
n_before = len(df_all_pw)
valid_mask = df_all_pw[PW_FEAT_COLS].notna().all(axis=1)
df_all_pw = df_all_pw[valid_mask].copy()
print(f"Dropped {n_before - len(df_all_pw)} invalid rows, {len(df_all_pw)} beats remaining")

# --- P-wave template correlation ---
def compute_pwave_template_corr(df, template_len=PW_TEMPLATE_LEN):
    """Per-patient P-wave template from N beats, correlate all beats against it."""
    df = df.copy()
    df['pw_template_corr'] = np.nan
    
    for rec in df['record'].unique():
        rec_mask = df['record'] == rec
        rec_df = df[rec_mask]
        
        # N-beat P-wave waveforms for template
        n_waveforms = []
        for wf in rec_df.loc[rec_df['clinical_label'] == 'N', 'pw_waveform']:
            if wf is not None and hasattr(wf, '__len__') and len(wf) == template_len:
                n_waveforms.append(wf)
        
        if len(n_waveforms) < 10:
            # Fallback: accept any waveform close to template_len and resample
            n_waveforms = []
            for wf in rec_df.loc[rec_df['clinical_label'] == 'N', 'pw_waveform']:
                if wf is not None and hasattr(wf, '__len__') and len(wf) >= 5:
                    n_waveforms.append(sig_resample(wf, template_len))
            if len(n_waveforms) < 10:
                continue
        
        template = np.mean(n_waveforms, axis=0)
        template_centered = template - np.mean(template)
        template_std = np.std(template_centered)
        if template_std == 0:
            continue
        
        corr_values = []
        indices = []
        for idx, wf in zip(rec_df.index, rec_df['pw_waveform']):
            if wf is not None and hasattr(wf, '__len__') and len(wf) >= 5:
                if len(wf) != template_len:
                    wf = sig_resample(wf, template_len)
                beat_centered = wf - np.mean(wf)
                beat_std = np.std(beat_centered)
                if beat_std > 0:
                    corr = np.corrcoef(template_centered, beat_centered)[0, 1]
                else:
                    corr = 0.0
            else:
                corr = 0.0
            corr_values.append(corr)
            indices.append(idx)
        
        df.loc[indices, 'pw_template_corr'] = corr_values
    
    return df

df_all_pw = compute_pwave_template_corr(df_all_pw)
print(f"Template correlation NaNs: {df_all_pw['pw_template_corr'].isna().sum()}")

# Fill remaining NaNs with 0 (records with too few N beats for template)
df_all_pw['pw_template_corr'] = df_all_pw['pw_template_corr'].fillna(0.0)
print(f"After template correlation: {len(df_all_pw)} beats")

# --- Per-patient normalization ---
for feat in PW_FEAT_COLS:
    n_beat_mask = df_all_pw['clinical_label'] == 'N'
    n_stats = df_all_pw[n_beat_mask].groupby('record')[feat].agg(['mean', 'std'])
    n_stats['std'] = n_stats['std'].replace(0, 1)
    
    df_all_pw[f'{feat}_pnorm'] = np.nan
    for rec in df_all_pw['record'].unique():
        if rec in n_stats.index:
            rec_mask = df_all_pw['record'] == rec
            df_all_pw.loc[rec_mask, f'{feat}_pnorm'] = (
                (df_all_pw.loc[rec_mask, feat] - n_stats.loc[rec, 'mean']) / n_stats.loc[rec, 'std']
            )

PW_PNORM_COLS = [f'{feat}_pnorm' for feat in PW_FEAT_COLS]

# Drop any NaNs from normalization
n_before = len(df_all_pw)
df_all_pw = df_all_pw.dropna(subset=PW_PNORM_COLS).copy()
print(f"Dropped {n_before - len(df_all_pw)} NaN rows from normalization")
print(f"Final dataset: {len(df_all_pw)} beats")

# --- Split ---
train_mask_pw = df_all_pw['record'].isin(DS1_RECORDS)
test_mask_pw = df_all_pw['record'].isin(DS2_RECORDS)
df_train_pw = df_all_pw[train_mask_pw].copy()
df_test_pw = df_all_pw[test_mask_pw].copy()
y_train_pw = df_train_pw['clinical_label'].values
y_test_pw = df_test_pw['clinical_label'].values
print(f"Train: {len(df_train_pw)}, Test: {len(df_test_pw)}")

In [ ]:
# --- Cohen's d analysis + P-wave window visualization ---

ALL_PW_FEATURES = PW_FEAT_COLS + PW_PNORM_COLS + ['pw_template_corr']

# Cohen's d: raw, N-beat-normalized, and template corr
print("=== P-Wave Feature Separation (Cohen's d, N vs S) ===\n")
print(f"{'Feature':<30} {'Raw d':>8} {'N-norm d':>8} {'Change':>8}")
print("-" * 58)

for feat in PW_FEAT_COLS:
    n_vals = df_test_pw.loc[df_test_pw['clinical_label'] == 'N', feat]
    s_vals = df_test_pw.loc[df_test_pw['clinical_label'] == 'S', feat]
    pooled = np.sqrt((n_vals.std()**2 + s_vals.std()**2) / 2)
    raw_d = abs(n_vals.mean() - s_vals.mean()) / pooled if pooled > 0 else 0

    feat_pn = f'{feat}_pnorm'
    n_pn = df_test_pw.loc[df_test_pw['clinical_label'] == 'N', feat_pn]
    s_pn = df_test_pw.loc[df_test_pw['clinical_label'] == 'S', feat_pn]
    pooled_pn = np.sqrt((n_pn.std()**2 + s_pn.std()**2) / 2)
    pnorm_d = abs(n_pn.mean() - s_pn.mean()) / pooled_pn if pooled_pn > 0 else 0

    change = pnorm_d - raw_d
    marker = " ***" if pnorm_d > 0.3 else ""
    print(f"{feat:<30} {raw_d:>8.3f} {pnorm_d:>8.3f} {change:>+8.3f}{marker}")

# Template correlation
n_corr = df_test_pw.loc[df_test_pw['clinical_label'] == 'N', 'pw_template_corr']
s_corr = df_test_pw.loc[df_test_pw['clinical_label'] == 'S', 'pw_template_corr']
v_corr = df_test_pw.loc[df_test_pw['clinical_label'] == 'V', 'pw_template_corr']
pooled_tc = np.sqrt((n_corr.std()**2 + s_corr.std()**2) / 2)
tc_d = abs(n_corr.mean() - s_corr.mean()) / pooled_tc if pooled_tc > 0 else 0
print(f"\n{'pw_template_corr':<30} {'—':>8} {tc_d:>8.3f} {'(new)':>8}")
print(f"  N mean={n_corr.mean():.3f} (std={n_corr.std():.3f})")
print(f"  S mean={s_corr.mean():.3f} (std={s_corr.std():.3f})")
print(f"  V mean={v_corr.mean():.3f} (std={v_corr.std():.3f})")

# Visualize P-wave template correlation + key features
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Template correlation distribution
ax = axes[0, 0]
for label, color in [('N', 'steelblue'), ('S', 'orange'), ('V', 'red')]:
    data = df_test_pw.loc[df_test_pw['clinical_label'] == label, 'pw_template_corr']
    ax.hist(data, bins=60, alpha=0.5, label=f'{label} (n={len(data)})', color=color, density=True)
ax.set_title(f'P-wave Template Correlation (d={tc_d:.3f})')
ax.set_xlabel('Correlation')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Top 3 N-normalized features by Cohen's d
top_feats = sorted(PW_PNORM_COLS, key=lambda f: abs(
    df_test_pw.loc[df_test_pw['clinical_label'] == 'N', f].mean() -
    df_test_pw.loc[df_test_pw['clinical_label'] == 'S', f].mean()
) / max(0.001, np.sqrt((df_test_pw.loc[df_test_pw['clinical_label'] == 'N', f].std()**2 +
        df_test_pw.loc[df_test_pw['clinical_label'] == 'S', f].std()**2) / 2)), reverse=True)[:3]

for ax, feat in zip([axes[0,1], axes[1,0], axes[1,1]], top_feats):
    for label, color in [('N', 'steelblue'), ('S', 'orange'), ('V', 'red')]:
        data = df_test_pw.loc[df_test_pw['clinical_label'] == label, feat]
        ax.hist(data, bins=50, alpha=0.5, label=label, color=color, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('P-Wave Window Features — Class Distributions (DS2)', fontsize=13)
plt.tight_layout()
plt.show()

# Check P-wave window on specific records
print("\n=== P-wave template correlation by record (S-bearing records) ===\n")
print(f"{'Record':<8} {'S beats':>8} {'N pw_corr':>10} {'S pw_corr':>10} {'Diff':>8}")
print("-" * 48)
for rec in sorted(df_test_pw['record'].unique()):
    rec_df = df_test_pw[df_test_pw['record'] == rec]
    n_s = (rec_df['clinical_label'] == 'S').sum()
    if n_s == 0:
        continue
    n_corr_rec = rec_df.loc[rec_df['clinical_label'] == 'N', 'pw_template_corr'].mean()
    s_corr_rec = rec_df.loc[rec_df['clinical_label'] == 'S', 'pw_template_corr'].mean()
    diff = s_corr_rec - n_corr_rec
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {n_s:>8} {n_corr_rec:>10.3f} {s_corr_rec:>10.3f} {diff:>+8.3f}{marker}")

In [ ]:
# --- Train GB with P-wave features and compare ---

# Full set: 34 original + 9 raw P-wave + 9 N-normalized P-wave + 1 template corr = 53
FEATURES_PW_ALL = FEATURE_COLS + PW_FEAT_COLS + PW_PNORM_COLS + ['pw_template_corr']
print(f"Full P-wave feature set: {len(FEATURES_PW_ALL)} features\n")

# Train GB
sw_pw = compute_sample_weight('balanced', y_train_pw)
clf_gb_pw = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_pw.fit(df_train_pw[FEATURES_PW_ALL].values, y_train_pw, sample_weight=sw_pw)
y_pred_pw = clf_gb_pw.predict(df_test_pw[FEATURES_PW_ALL].values)

# Baseline on matched dataset
clf_gb_base_pw = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_base_pw.fit(df_train_pw[FEATURE_COLS].values, y_train_pw, sample_weight=sw_pw)
y_pred_base_pw = clf_gb_base_pw.predict(df_test_pw[FEATURE_COLS].values)

print("=== Baseline GB (34 features, matched dataset) ===")
print(classification_report(y_test_pw, y_pred_base_pw, labels=labels_3, digits=3, zero_division=0))

print(f"=== GB + P-wave features ({len(FEATURES_PW_ALL)} features) ===")
print(classification_report(y_test_pw, y_pred_pw, labels=labels_3, digits=3, zero_division=0))

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_base = confusion_matrix(y_test_pw, y_pred_base_pw, labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_base, display_labels=labels_3).plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Baseline GB (34)')

cm_pw = confusion_matrix(y_test_pw, y_pred_pw, labels=labels_3)
ConfusionMatrixDisplay(confusion_matrix=cm_pw, display_labels=labels_3).plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title(f'GB + P-wave ({len(FEATURES_PW_ALL)})')

plt.suptitle('P-Wave Presence Features — Impact on Classification', fontsize=13)
plt.tight_layout()
plt.show()

# Subset analysis
test_records_pw = df_test_pw['record'].values
mask_232_pw = test_records_pw == '232'
mask_not232_pw = ~mask_232_pw

print("\n=== Subset Comparison ===\n")
print(f"{'Subset':<16} {'Metric':<10} {'Baseline (34)':>14} {'+ P-wave':>14} {'Delta':>8}")
print("-" * 66)

for subset_name, mask in [('All DS2', np.ones(len(y_test_pw), dtype=bool)),
                           ('Without 232', mask_not232_pw),
                           ('232 only', mask_232_pw)]:
    for metric_name, metric_fn in [('S F1', lambda yt, yp: f1_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('V F1', lambda yt, yp: f1_score(yt, yp, labels=['V'], average=None, zero_division=0)[0]),
                                    ('Macro F1', lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0))]:
        base_val = metric_fn(y_test_pw[mask], y_pred_base_pw[mask])
        pw_val = metric_fn(y_test_pw[mask], y_pred_pw[mask])
        delta = pw_val - base_val
        print(f"{subset_name:<16} {metric_name:<10} {base_val:>14.3f} {pw_val:>14.3f} {delta:>+8.3f}")
    print()

# Per-record S F1
print("=== Per-record S F1: Baseline vs + P-wave ===\n")
print(f"{'Record':<8} {'S beats':>8} {'Base S F1':>10} {'PW S F1':>10} {'Delta':>8}")
print("-" * 48)
for rec in sorted(df_test_pw['record'].unique()):
    mask_rec = test_records_pw == rec
    true_s = (y_test_pw[mask_rec] == 'S').sum()
    if true_s == 0:
        continue
    s_base = f1_score(y_test_pw[mask_rec], y_pred_base_pw[mask_rec], labels=['S'], average=None, zero_division=0)[0]
    s_pw = f1_score(y_test_pw[mask_rec], y_pred_pw[mask_rec], labels=['S'], average=None, zero_division=0)[0]
    delta = s_pw - s_base
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {true_s:>8} {s_base:>10.3f} {s_pw:>10.3f} {delta:>+8.3f}{marker}")

In [ ]:
# --- Adaptive P-wave window + P-wave PCA ---
# Improvement 1: Scale P-wave window as proportion of preceding RR interval
# Improvement 2: PCA on P-wave waveforms (fit on DS1 N beats)

from scipy.signal import find_peaks, resample as sig_resample
from sklearn.decomposition import PCA

PW_RESAMPLE_LEN = 44  # fixed resampled length for PCA/template (~120ms at 360 Hz)
N_PW_PCA = 5  # fewer components than QRS PCA — P-wave is simpler

def extract_pwave_features_v2(data_dir, rec_id, df_rec, fs=360):
    """Extract P-wave features using adaptive window proportional to RR interval.
    
    Window: 25% to 10% of preceding RR interval before R-peak.
    At 75 bpm (RR=800ms): 200ms to 80ms (same as v1).
    At 150 bpm (RR=400ms): 100ms to 40ms (stays in P-wave region).
    For premature S beats: window shrinks proportionally, avoiding T-wave overlap.
    """
    record = wfdb.rdrecord(f'{data_dir}/{rec_id}')
    _, filtered = pan_tompkins_detect(record.p_signal[:, 0], record.fs)
    
    sample_indices = df_rec['sample_idx'].values
    features = []
    
    nan_row = {
        'pw2_peak_amplitude': np.nan, 'pw2_energy': np.nan,
        'pw2_peak_prominence': np.nan, 'pw2_has_peak': np.nan,
        'pw2_baseline_dev': np.nan, 'pw2_max': np.nan,
        'pw2_min': np.nan, 'pw2_range': np.nan, 'pw2_area': np.nan,
    }
    
    for idx, row in df_rec.iterrows():
        r_curr = int(row['sample_idx'])
        
        # Find preceding R-peak for adaptive window
        pos = np.searchsorted(sample_indices, r_curr)
        if pos > 0:
            r_prev = int(sample_indices[pos - 1])
            rr_samples = r_curr - r_prev
        else:
            rr_samples = int(0.8 * fs)  # default 800ms if no previous beat
        
        # Adaptive window: 25% to 10% of RR interval
        pw_start = r_curr - int(0.25 * rr_samples)
        pw_end = r_curr - int(0.10 * rr_samples)
        
        # Safety: clip to not overlap previous QRS (100ms after prev R-peak)
        if pos > 0:
            prev_qrs_end = r_prev + int(0.100 * fs)
            if pw_start < prev_qrs_end:
                pw_start = prev_qrs_end
        
        # Minimum window: 15ms (too short = meaningless)
        min_window = int(0.015 * fs)
        if pw_start < 0 or pw_end >= len(filtered) or pw_end <= pw_start or (pw_end - pw_start) < min_window:
            feat = nan_row.copy()
            feat['sample_idx'] = r_curr
            feat['pw2_waveform'] = None
            features.append(feat)
            continue
        
        pw_segment = filtered[pw_start:pw_end]
        
        # Baseline: average of first and last 3 samples
        n_edge = min(3, len(pw_segment) // 3)
        if n_edge < 1:
            n_edge = 1
        baseline = (np.mean(pw_segment[:n_edge]) + np.mean(pw_segment[-n_edge:])) / 2
        pw_detrended = pw_segment - baseline
        
        # Peak detection
        peaks, properties = find_peaks(pw_detrended, prominence=0.01)
        has_peak = len(peaks) > 0
        if has_peak:
            best_peak = peaks[np.argmax(properties['prominences'])]
            peak_amp = pw_detrended[best_peak]
            peak_prom = properties['prominences'][np.argmax(properties['prominences'])]
        else:
            peak_amp = np.max(pw_detrended)
            peak_prom = 0.0
        
        # Resample to fixed length for PCA and template
        pw_resampled = sig_resample(pw_segment, PW_RESAMPLE_LEN)
        
        features.append({
            'sample_idx': r_curr,
            'pw2_waveform': pw_resampled,
            'pw2_peak_amplitude': peak_amp,
            'pw2_energy': np.sum(pw_detrended ** 2) / len(pw_detrended),
            'pw2_peak_prominence': peak_prom,
            'pw2_has_peak': float(has_peak),
            'pw2_baseline_dev': np.mean(np.abs(pw_detrended)),
            'pw2_max': np.max(pw_segment),
            'pw2_min': np.min(pw_segment),
            'pw2_range': np.max(pw_segment) - np.min(pw_segment),
            'pw2_area': np.trapezoid(pw_detrended) / fs,
        })
    
    return pd.DataFrame(features)

# --- Extract for all records ---
pwave2_dfs = []
for rec_id in sorted(df_all['record'].unique()):
    df_rec = df_all[df_all['record'] == rec_id]
    pw_feats = extract_pwave_features_v2(DATA_DIR, rec_id, df_rec)
    pwave2_dfs.append(pw_feats)
    n_valid = pw_feats['pw2_energy'].notna().sum()
    n_invalid = len(pw_feats) - n_valid
    if n_invalid > 0:
        print(f"Record {rec_id}: {len(pw_feats)} beats, {n_invalid} invalid")

df_pwave2 = pd.concat(pwave2_dfs, ignore_index=True)

PW2_FEAT_COLS = ['pw2_peak_amplitude', 'pw2_energy', 'pw2_peak_prominence',
                 'pw2_has_peak', 'pw2_baseline_dev', 'pw2_max', 'pw2_min',
                 'pw2_range', 'pw2_area']

n_valid = df_pwave2['pw2_energy'].notna().sum()
print(f"\nExtracted P-wave v2 features for {n_valid}/{len(df_pwave2)} beats ({len(df_pwave2)-n_valid} invalid)")


In [ ]:
# --- Template correlation + PCA + per-patient normalization ---

# Merge into df_all
df_all_pw2 = df_all.copy().reset_index(drop=True)
df_pwave2_aligned = df_pwave2.reset_index(drop=True)

for col in PW2_FEAT_COLS:
    df_all_pw2[col] = df_pwave2_aligned[col].values
df_all_pw2['pw2_waveform'] = df_pwave2_aligned['pw2_waveform'].values

# Drop invalid rows
n_before = len(df_all_pw2)
valid_mask = df_all_pw2[PW2_FEAT_COLS].notna().all(axis=1)
df_all_pw2 = df_all_pw2[valid_mask].copy()
print(f"Dropped {n_before - len(df_all_pw2)} invalid rows, {len(df_all_pw2)} beats remaining")

# --- Per-patient P-wave template correlation ---
def compute_pwave_template_corr_v2(df):
    df = df.copy()
    df['pw2_template_corr'] = np.nan
    
    for rec in df['record'].unique():
        rec_mask = df['record'] == rec
        rec_df = df[rec_mask]
        
        n_waveforms = []
        for wf in rec_df.loc[rec_df['clinical_label'] == 'N', 'pw2_waveform']:
            if wf is not None and hasattr(wf, '__len__') and len(wf) == PW_RESAMPLE_LEN:
                n_waveforms.append(wf)
            elif wf is not None and hasattr(wf, '__len__') and len(wf) >= 5:
                n_waveforms.append(sig_resample(wf, PW_RESAMPLE_LEN))
        
        if len(n_waveforms) < 10:
            continue
        
        template = np.mean(n_waveforms, axis=0)
        template_centered = template - np.mean(template)
        template_std = np.std(template_centered)
        if template_std == 0:
            continue
        
        corr_values = []
        indices = []
        for idx, wf in zip(rec_df.index, rec_df['pw2_waveform']):
            if wf is not None and hasattr(wf, '__len__') and len(wf) >= 5:
                if len(wf) != PW_RESAMPLE_LEN:
                    wf = sig_resample(wf, PW_RESAMPLE_LEN)
                beat_centered = wf - np.mean(wf)
                beat_std = np.std(beat_centered)
                if beat_std > 0:
                    corr = np.corrcoef(template_centered, beat_centered)[0, 1]
                else:
                    corr = 0.0
            else:
                corr = 0.0
            corr_values.append(corr)
            indices.append(idx)
        
        df.loc[indices, 'pw2_template_corr'] = corr_values
    
    df['pw2_template_corr'] = df['pw2_template_corr'].fillna(0.0)
    return df

df_all_pw2 = compute_pwave_template_corr_v2(df_all_pw2)
print(f"Template correlation computed: {(df_all_pw2['pw2_template_corr'] != 0).sum()} non-zero values")

# --- P-wave PCA (fit on DS1 N beats only) ---
ds1_mask = df_all_pw2['record'].isin([str(r) for r in DS1_RECORDS])
n_mask = df_all_pw2['clinical_label'] == 'N'

# Collect DS1 N-beat waveforms
ds1_n_waveforms = []
for wf in df_all_pw2.loc[ds1_mask & n_mask, 'pw2_waveform']:
    if wf is not None and hasattr(wf, '__len__'):
        if len(wf) == PW_RESAMPLE_LEN:
            ds1_n_waveforms.append(wf)
        elif len(wf) >= 5:
            ds1_n_waveforms.append(sig_resample(wf, PW_RESAMPLE_LEN))

ds1_n_matrix = np.array(ds1_n_waveforms)
print(f"PCA training set: {ds1_n_matrix.shape[0]} DS1 N-beat P-wave waveforms ({PW_RESAMPLE_LEN} samples each)")

pw_pca = PCA(n_components=N_PW_PCA, random_state=42)
pw_pca.fit(ds1_n_matrix)
print(f"P-wave PCA variance explained: {pw_pca.explained_variance_ratio_.sum():.1%} ({N_PW_PCA} components)")
for i, v in enumerate(pw_pca.explained_variance_ratio_):
    print(f"  pw_pca_{i}: {v:.1%}")

# Apply PCA to all beats
pw_pca_cols = [f'pw_pca_{i}' for i in range(N_PW_PCA)]
all_waveforms = []
valid_indices = []
for idx, wf in zip(df_all_pw2.index, df_all_pw2['pw2_waveform']):
    if wf is not None and hasattr(wf, '__len__'):
        if len(wf) == PW_RESAMPLE_LEN:
            all_waveforms.append(wf)
            valid_indices.append(idx)
        elif len(wf) >= 5:
            all_waveforms.append(sig_resample(wf, PW_RESAMPLE_LEN))
            valid_indices.append(idx)

all_wf_matrix = np.array(all_waveforms)
pca_transformed = pw_pca.transform(all_wf_matrix)

for i, col in enumerate(pw_pca_cols):
    df_all_pw2[col] = np.nan
    df_all_pw2.loc[valid_indices, col] = pca_transformed[:, i]

# Fill NaN PCA values with 0
for col in pw_pca_cols:
    df_all_pw2[col] = df_all_pw2[col].fillna(0.0)

# --- Per-patient normalization of P-wave v2 features ---
for feat in PW2_FEAT_COLS:
    n_stats = df_all_pw2[n_mask | (~ds1_mask & (df_all_pw2['clinical_label'] == 'N'))].groupby('record')[feat].agg(['mean', 'std'])
    # Use all N beats for normalization stats
    n_stats_all = df_all_pw2[df_all_pw2['clinical_label'] == 'N'].groupby('record')[feat].agg(['mean', 'std'])
    n_stats_all['std'] = n_stats_all['std'].replace(0, 1)
    
    df_all_pw2[f'{feat}_pnorm'] = np.nan
    for rec in df_all_pw2['record'].unique():
        if rec in n_stats_all.index:
            rec_mask_r = df_all_pw2['record'] == rec
            df_all_pw2.loc[rec_mask_r, f'{feat}_pnorm'] = (
                (df_all_pw2.loc[rec_mask_r, feat] - n_stats_all.loc[rec, 'mean']) / n_stats_all.loc[rec, 'std']
            )

PW2_PNORM_COLS = [f'{feat}_pnorm' for feat in PW2_FEAT_COLS]

# Drop normalization NaNs
n_before = len(df_all_pw2)
df_all_pw2 = df_all_pw2.dropna(subset=PW2_PNORM_COLS + pw_pca_cols).copy()
print(f"Dropped {n_before - len(df_all_pw2)} NaN rows, final: {len(df_all_pw2)} beats")

# --- Train/test split ---
train_mask_pw2 = df_all_pw2['record'].isin([str(r) for r in DS1_RECORDS])
df_train_pw2 = df_all_pw2[train_mask_pw2]
df_test_pw2 = df_all_pw2[~train_mask_pw2]
y_train_pw2 = df_train_pw2['clinical_label'].values
y_test_pw2 = df_test_pw2['clinical_label'].values
print(f"Train: {len(df_train_pw2)}, Test: {len(df_test_pw2)}")


In [ ]:
# --- Cohen's d analysis for adaptive P-wave v2 features ---

ALL_PW2_FEATURES = PW2_FEAT_COLS + PW2_PNORM_COLS + ['pw2_template_corr'] + pw_pca_cols

print("=== Adaptive P-Wave v2 Feature Separation (Cohen's d, N vs S) ===\n")
print(f"{'Feature':<35} {'Raw d':>8} {'N-norm d':>8} {'Change':>8}")
print("-" * 63)

for feat in PW2_FEAT_COLS:
    n_vals = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'N', feat]
    s_vals = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'S', feat]
    pooled = np.sqrt((n_vals.std()**2 + s_vals.std()**2) / 2)
    raw_d = abs(n_vals.mean() - s_vals.mean()) / pooled if pooled > 0 else 0

    feat_pn = f'{feat}_pnorm'
    n_pn = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'N', feat_pn]
    s_pn = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'S', feat_pn]
    pooled_pn = np.sqrt((n_pn.std()**2 + s_pn.std()**2) / 2)
    pnorm_d = abs(n_pn.mean() - s_pn.mean()) / pooled_pn if pooled_pn > 0 else 0

    change = pnorm_d - raw_d
    marker = " ***" if pnorm_d > 0.3 else ""
    print(f"{feat:<35} {raw_d:>8.3f} {pnorm_d:>8.3f} {change:>+8.3f}{marker}")

# Template correlation
n_corr = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'N', 'pw2_template_corr']
s_corr = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'S', 'pw2_template_corr']
v_corr = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'V', 'pw2_template_corr']
pooled_tc = np.sqrt((n_corr.std()**2 + s_corr.std()**2) / 2)
tc_d = abs(n_corr.mean() - s_corr.mean()) / pooled_tc if pooled_tc > 0 else 0
print(f"\n{'pw2_template_corr':<35} {'—':>8} {tc_d:>8.3f} {'(new)':>8}")
print(f"  N mean={n_corr.mean():.3f} (std={n_corr.std():.3f})")
print(f"  S mean={s_corr.mean():.3f} (std={s_corr.std():.3f})")
print(f"  V mean={v_corr.mean():.3f} (std={v_corr.std():.3f})")

# PCA components
print(f"\nP-wave PCA components (Cohen's d, N vs S):")
for col in pw_pca_cols:
    n_pca = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'N', col]
    s_pca = df_test_pw2.loc[df_test_pw2['clinical_label'] == 'S', col]
    pooled_p = np.sqrt((n_pca.std()**2 + s_pca.std()**2) / 2)
    d_pca = abs(n_pca.mean() - s_pca.mean()) / pooled_p if pooled_p > 0 else 0
    marker = " ***" if d_pca > 0.3 else ""
    print(f"  {col:<33} {d_pca:>8.3f}{marker}")

# Compare v1 vs v2 template correlation
print(f"\n=== v1 (fixed window) vs v2 (adaptive window) template correlation ===\n")
print(f"{'Metric':<25} {'v1 (fixed)':>12} {'v2 (adaptive)':>14}")
print("-" * 53)

# v1 values from previous cell output
v1_n = df_test_pw.loc[df_test_pw['clinical_label'] == 'N', 'pw_template_corr']
v1_s = df_test_pw.loc[df_test_pw['clinical_label'] == 'S', 'pw_template_corr']
v1_pooled = np.sqrt((v1_n.std()**2 + v1_s.std()**2) / 2)
v1_d = abs(v1_n.mean() - v1_s.mean()) / v1_pooled if v1_pooled > 0 else 0

print(f"{'N mean corr':<25} {v1_n.mean():>12.3f} {n_corr.mean():>14.3f}")
print(f"{'S mean corr':<25} {v1_s.mean():>12.3f} {s_corr.mean():>14.3f}")
print(f"{'Cohen d (N vs S)':<25} {v1_d:>12.3f} {tc_d:>14.3f}")

# Per-record comparison
print(f"\n=== Per-record template correlation: v1 vs v2 ===\n")
print(f"{'Record':<8} {'S beats':>8} {'v1 N':>8} {'v1 S':>8} {'v2 N':>8} {'v2 S':>8} {'v1 diff':>8} {'v2 diff':>8}")
print("-" * 72)

for rec in sorted(df_test_pw2['record'].unique()):
    rec_df2 = df_test_pw2[df_test_pw2['record'] == rec]
    n_s = (rec_df2['clinical_label'] == 'S').sum()
    if n_s == 0:
        continue
    v2_n_corr = rec_df2.loc[rec_df2['clinical_label'] == 'N', 'pw2_template_corr'].mean()
    v2_s_corr = rec_df2.loc[rec_df2['clinical_label'] == 'S', 'pw2_template_corr'].mean()
    
    rec_df1 = df_test_pw[df_test_pw['record'] == rec]
    v1_n_c = rec_df1.loc[rec_df1['clinical_label'] == 'N', 'pw_template_corr'].mean()
    v1_s_c = rec_df1.loc[rec_df1['clinical_label'] == 'S', 'pw_template_corr'].mean()
    
    v1_diff = v1_n_c - v1_s_c
    v2_diff = v2_n_corr - v2_s_corr
    marker = " <<<" if rec == '232' else (" !!" if v2_diff < v1_diff - 0.1 else "")
    print(f"{rec:<8} {n_s:>8} {v1_n_c:>8.3f} {v1_s_c:>8.3f} {v2_n_corr:>8.3f} {v2_s_corr:>8.3f} {v1_diff:>+8.3f} {v2_diff:>+8.3f}{marker}")


In [ ]:
# --- Train GB with adaptive P-wave v2 + PCA features ---

# Feature sets to compare:
# A: 34 original (baseline)
# B: 34 + 9 raw pw2 + 9 pnorm pw2 + 1 template_corr + 5 PCA = 58 features
# C: 34 + only the features with d > 0.3 + template_corr + PCA (curated)

FEATURES_PW2_ALL = FEATURE_COLS + PW2_FEAT_COLS + PW2_PNORM_COLS + ['pw2_template_corr'] + pw_pca_cols
print(f"Full adaptive P-wave v2 feature set: {len(FEATURES_PW2_ALL)} features\n")

# --- Train full model (58 features) ---
sw_pw2 = compute_sample_weight('balanced', y_train_pw2)
clf_gb_pw2 = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_pw2.fit(df_train_pw2[FEATURES_PW2_ALL].values, y_train_pw2, sample_weight=sw_pw2)
y_pred_pw2 = clf_gb_pw2.predict(df_test_pw2[FEATURES_PW2_ALL].values)

# --- Baseline on matched dataset ---
clf_gb_base2 = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_base2.fit(df_train_pw2[FEATURE_COLS].values, y_train_pw2, sample_weight=sw_pw2)
y_pred_base2 = clf_gb_base2.predict(df_test_pw2[FEATURE_COLS].values)

# --- v1 P-wave on matched dataset (53 features) for fair comparison ---
# Use the v1 features that are already in df_all_pw
FEATURES_PW1_ALL = FEATURE_COLS + PW_FEAT_COLS + PW_PNORM_COLS + ['pw_template_corr']

# Align v1 features to pw2 dataset
df_test_pw2_with_v1 = df_test_pw2.copy()
df_train_pw2_with_v1 = df_train_pw2.copy()

# Check if v1 features exist in df_all_pw and align by index
pw1_available = all(col in df_all_pw.columns for col in FEATURES_PW1_ALL)
if pw1_available:
    # Match on record + sample_idx
    for col in PW_FEAT_COLS + PW_PNORM_COLS + ['pw_template_corr']:
        if col not in df_test_pw2_with_v1.columns:
            df_test_pw2_with_v1[col] = np.nan
            df_train_pw2_with_v1[col] = np.nan
    
    # Merge v1 features via record + sample_idx
    v1_cols = PW_FEAT_COLS + PW_PNORM_COLS + ['pw_template_corr']
    v1_merge = df_all_pw[['record', 'sample_idx'] + v1_cols].copy()
    
    df_train_pw2_with_v1 = df_train_pw2_with_v1.drop(columns=v1_cols, errors='ignore')
    df_train_pw2_with_v1 = df_train_pw2_with_v1.merge(v1_merge, on=['record', 'sample_idx'], how='left')
    
    df_test_pw2_with_v1 = df_test_pw2_with_v1.drop(columns=v1_cols, errors='ignore')
    df_test_pw2_with_v1 = df_test_pw2_with_v1.merge(v1_merge, on=['record', 'sample_idx'], how='left')
    
    # Fill NaN with 0 for any missing v1 features
    for col in v1_cols:
        df_train_pw2_with_v1[col] = df_train_pw2_with_v1[col].fillna(0.0)
        df_test_pw2_with_v1[col] = df_test_pw2_with_v1[col].fillna(0.0)
    
    clf_gb_pw1 = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.1,
        min_samples_leaf=20, l2_regularization=1.0, random_state=42,
    )
    clf_gb_pw1.fit(df_train_pw2_with_v1[FEATURES_PW1_ALL].values, y_train_pw2, sample_weight=sw_pw2)
    y_pred_pw1 = clf_gb_pw1.predict(df_test_pw2_with_v1[FEATURES_PW1_ALL].values)

print("=== Baseline GB (34 features, matched dataset) ===")
print(classification_report(y_test_pw2, y_pred_base2, labels=labels_3, digits=3, zero_division=0))

if pw1_available:
    print(f"=== GB + P-wave v1 fixed window ({len(FEATURES_PW1_ALL)} features) ===")
    print(classification_report(y_test_pw2, y_pred_pw1, labels=labels_3, digits=3, zero_division=0))

print(f"=== GB + P-wave v2 adaptive + PCA ({len(FEATURES_PW2_ALL)} features) ===")
print(classification_report(y_test_pw2, y_pred_pw2, labels=labels_3, digits=3, zero_division=0))

# --- Subset comparison ---
test_records_pw2 = df_test_pw2['record'].values
mask_232 = test_records_pw2 == '232'
mask_not232 = ~mask_232

print("\n=== Subset Comparison: Baseline vs v1 (fixed) vs v2 (adaptive+PCA) ===\n")
header = f"{'Subset':<16} {'Metric':<10} {'Base (34)':>10} {'v1 (53)':>10} {'v2 (58)':>10} {'v1 delta':>9} {'v2 delta':>9}"
print(header)
print("-" * len(header))

for subset_name, mask in [('All DS2', np.ones(len(y_test_pw2), dtype=bool)),
                           ('Without 232', mask_not232),
                           ('232 only', mask_232)]:
    for metric_name, metric_fn in [('S F1', lambda yt, yp: f1_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('V F1', lambda yt, yp: f1_score(yt, yp, labels=['V'], average=None, zero_division=0)[0]),
                                    ('Macro F1', lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0))]:
        base_val = metric_fn(y_test_pw2[mask], y_pred_base2[mask])
        v2_val = metric_fn(y_test_pw2[mask], y_pred_pw2[mask])
        if pw1_available:
            v1_val = metric_fn(y_test_pw2[mask], y_pred_pw1[mask])
        else:
            v1_val = float('nan')
        print(f"{subset_name:<16} {metric_name:<10} {base_val:>10.3f} {v1_val:>10.3f} {v2_val:>10.3f} {v1_val-base_val:>+9.3f} {v2_val-base_val:>+9.3f}")
    print()

# --- Per-record S F1 ---
print("=== Per-record S F1: Baseline vs v1 vs v2 ===\n")
print(f"{'Record':<8} {'S beats':>8} {'Base':>8} {'v1':>8} {'v2':>8} {'v1-base':>8} {'v2-base':>8}")
print("-" * 60)

for rec in sorted(df_test_pw2['record'].unique()):
    rec_mask = test_records_pw2 == rec
    n_s = (y_test_pw2[rec_mask] == 'S').sum()
    if n_s == 0:
        continue
    base_s = f1_score(y_test_pw2[rec_mask], y_pred_base2[rec_mask], labels=['S'], average=None, zero_division=0)[0]
    v2_s = f1_score(y_test_pw2[rec_mask], y_pred_pw2[rec_mask], labels=['S'], average=None, zero_division=0)[0]
    if pw1_available:
        v1_s = f1_score(y_test_pw2[rec_mask], y_pred_pw1[rec_mask], labels=['S'], average=None, zero_division=0)[0]
    else:
        v1_s = float('nan')
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {n_s:>8} {base_s:>8.3f} {v1_s:>8.3f} {v2_s:>8.3f} {v1_s-base_s:>+8.3f} {v2_s-base_s:>+8.3f}{marker}")


In [ ]:
# --- Hybrid P-wave features ---
# v1 fixed window: template correlation (stable on 232)
# v2 adaptive window: raw features + PCA (better anatomical accuracy)
# Combine both into one feature set

# Build hybrid dataset: start with v2 dataset (has adaptive features + PCA)
df_hybrid = df_all_pw2.copy()

# Merge v1 template correlation via record + sample_idx
v1_tc = df_all_pw[['record', 'sample_idx', 'pw_template_corr']].copy()
df_hybrid = df_hybrid.merge(v1_tc, on=['record', 'sample_idx'], how='left')
df_hybrid['pw_template_corr'] = df_hybrid['pw_template_corr'].fillna(0.0)

# Hybrid feature set:
# 34 original
# + 9 adaptive raw features (pw2_*)
# + 9 adaptive N-normalized features (pw2_*_pnorm)  
# + 1 FIXED window template correlation (pw_template_corr from v1)
# + 1 ADAPTIVE window template correlation (pw2_template_corr from v2)
# + 5 P-wave PCA components (from v2 adaptive window)
# = 59 features

FEATURES_HYBRID = (FEATURE_COLS + PW2_FEAT_COLS + PW2_PNORM_COLS + 
                   ['pw_template_corr', 'pw2_template_corr'] + pw_pca_cols)

print(f"Hybrid feature set: {len(FEATURES_HYBRID)} features")
print(f"  34 original + 9 adaptive raw + 9 adaptive pnorm + 2 template corr (fixed+adaptive) + 5 PCA")

# Verify no NaNs
n_nan = df_hybrid[FEATURES_HYBRID].isna().sum().sum()
print(f"  NaN values: {n_nan}")

# Train/test split
train_mask_h = df_hybrid['record'].isin([str(r) for r in DS1_RECORDS])
df_train_h = df_hybrid[train_mask_h]
df_test_h = df_hybrid[~train_mask_h]
y_train_h = df_train_h['clinical_label'].values
y_test_h = df_test_h['clinical_label'].values
print(f"  Train: {len(df_train_h)}, Test: {len(df_test_h)}")

# --- Train hybrid model ---
sw_h = compute_sample_weight('balanced', y_train_h)

clf_gb_hybrid = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_hybrid.fit(df_train_h[FEATURES_HYBRID].values, y_train_h, sample_weight=sw_h)
y_pred_hybrid = clf_gb_hybrid.predict(df_test_h[FEATURES_HYBRID].values)

# --- Baseline on matched dataset ---
clf_gb_base_h = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=20, l2_regularization=1.0, random_state=42,
)
clf_gb_base_h.fit(df_train_h[FEATURE_COLS].values, y_train_h, sample_weight=sw_h)
y_pred_base_h = clf_gb_base_h.predict(df_test_h[FEATURE_COLS].values)

# --- Results ---
print("\n=== Baseline GB (34 features) ===")
print(classification_report(y_test_h, y_pred_base_h, labels=labels_3, digits=3, zero_division=0))

print(f"=== Hybrid GB ({len(FEATURES_HYBRID)} features: adaptive raw/pnorm + fixed template + PCA) ===")
print(classification_report(y_test_h, y_pred_hybrid, labels=labels_3, digits=3, zero_division=0))

# --- Subset comparison: all four approaches ---
test_recs_h = df_test_h['record'].values
mask_232_h = test_recs_h == '232'
mask_not232_h = ~mask_232_h

print("\n=== All Approaches Compared ===\n")
header = f"{'Subset':<16} {'Metric':<10} {'Base(34)':>9} {'v1(53)':>9} {'v2(58)':>9} {'Hybrid':>9}"
print(header)
print("-" * len(header))

for subset_name, mask in [('All DS2', np.ones(len(y_test_h), dtype=bool)),
                           ('Without 232', mask_not232_h),
                           ('232 only', mask_232_h)]:
    for metric_name, metric_fn in [('S F1', lambda yt, yp: f1_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('S Prec', lambda yt, yp: precision_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('S Rec', lambda yt, yp: recall_score(yt, yp, labels=['S'], average=None, zero_division=0)[0]),
                                    ('V F1', lambda yt, yp: f1_score(yt, yp, labels=['V'], average=None, zero_division=0)[0]),
                                    ('Macro F1', lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0))]:
        base_val = metric_fn(y_test_h[mask], y_pred_base_h[mask])
        hybrid_val = metric_fn(y_test_h[mask], y_pred_hybrid[mask])
        # v1 and v2 from previous cells (use y_pred_pw1 and y_pred_pw2 if same test set)
        v1_val = metric_fn(y_test_pw2[mask], y_pred_pw1[mask]) if pw1_available else float('nan')
        v2_val = metric_fn(y_test_pw2[mask], y_pred_pw2[mask])
        print(f"{subset_name:<16} {metric_name:<10} {base_val:>9.3f} {v1_val:>9.3f} {v2_val:>9.3f} {hybrid_val:>9.3f}")
    print()

# --- Per-record S F1 ---
print("=== Per-record S F1: All Approaches ===\n")
print(f"{'Record':<8} {'S beats':>8} {'Base':>8} {'v1':>8} {'v2':>8} {'Hybrid':>8} {'H-Base':>8}")
print("-" * 64)

for rec in sorted(df_test_h['record'].unique()):
    rec_mask_h = test_recs_h == rec
    n_s = (y_test_h[rec_mask_h] == 'S').sum()
    if n_s == 0:
        continue
    base_s = f1_score(y_test_h[rec_mask_h], y_pred_base_h[rec_mask_h], labels=['S'], average=None, zero_division=0)[0]
    hybrid_s = f1_score(y_test_h[rec_mask_h], y_pred_hybrid[rec_mask_h], labels=['S'], average=None, zero_division=0)[0]
    
    rec_mask_pw2 = df_test_pw2['record'].values == rec
    v1_s = f1_score(y_test_pw2[rec_mask_pw2], y_pred_pw1[rec_mask_pw2], labels=['S'], average=None, zero_division=0)[0] if pw1_available else float('nan')
    v2_s = f1_score(y_test_pw2[rec_mask_pw2], y_pred_pw2[rec_mask_pw2], labels=['S'], average=None, zero_division=0)[0]
    
    marker = " <<<" if rec == '232' else ""
    print(f"{rec:<8} {n_s:>8} {base_s:>8.3f} {v1_s:>8.3f} {v2_s:>8.3f} {hybrid_s:>8.3f} {hybrid_s-base_s:>+8.3f}{marker}")


# P-Wave Template Label Leakage Experiment

**Question:** Does using ground-truth N labels to build per-patient P-wave templates
constitute label leakage that inflates evaluation metrics?

`build_df_all()` builds P-wave templates from ground-truth N beats for ALL records,
including DS2 test records. At deployment, we won't have labels — we need to know
how much performance degrades with realistic template construction.

## Experimental Conditions

| Condition | DS1 templates | DS2 templates | Simulates |
|-----------|--------------|--------------|------------|
| A: Oracle | GT N labels | GT N labels | Current results (upper bound) |
| B: Deployment | GT N labels | First-pass classifier predictions | Realistic deployment |
| C: Unsupervised | All-beat mean | All-beat mean | No labels at all |
| D: RR-filtered | GT N labels | On-time beats (RR >= 80% median) | Label-free deployment |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from scipy.signal import resample as sig_resample

from ecg_monitor.pipeline import (
    build_df_all, get_train_test_split, compute_pwave_template_corr,
    _build_pwave_template, select_on_time_beats,
    FEATURE_COLS, FEATURE_COLS_48, FEATURE_COLS_59,
    PW2_RAW_COLS, PW2_PNORM_COLS, PW_PCA_COLS,
    PW_RESAMPLE_LEN, DS1_RECORDS, DS2_RECORDS,
)

In [ ]:
# Load data with waveforms retained for re-templating
df_all, pca, pw_pca = build_df_all(keep_waveforms=True)
df_train, df_test = get_train_test_split(df_all)

In [ ]:
# Verify P-wave waveform columns are present for re-templating
pw_waveform_cols = ['_pw_fixed_waveform', '_pw2_waveform']
has_waveforms = all(c in df_all.columns for c in pw_waveform_cols)
print(f"P-wave waveform columns available: {has_waveforms}")
if has_waveforms:
    print(f"  _pw_fixed_waveform non-null: {df_all['_pw_fixed_waveform'].notna().sum()}")
    print(f"  _pw2_waveform non-null: {df_all['_pw2_waveform'].notna().sum()}")
else:
    # If build_df_all drops waveform columns, we need to check
    print("WARNING: Waveform columns not in df_all — checking if dropped after template computation")
    print(f"Available columns: {sorted(df_all.columns.tolist())}")

## Helper: Recompute template features with different label sources

We need to rebuild `pw_template_corr` and `pw2_template_corr` on DS2 records
using different methods to select which beats form the template.

In [ ]:
def recompute_template_corr(df, record_col, wf_col, label_source, min_beats=10):
    """Recompute template correlation using a specified label source.

    Args:
        df: DataFrame with waveform column and record column
        record_col: column name for record ID
        wf_col: column name for P-wave waveform ('_pw_fixed_waveform' or '_pw2_waveform')
        label_source: one of:
            - 'gt': use ground-truth clinical_label == 'N'
            - 'all': use all beats (unsupervised)
            - 'rr_filtered': use on-time beats (RR >= 80% of median), no labels
            - array-like: predicted labels, use entries == 'N'
        min_beats: minimum N beats to form template

    Returns:
        np.ndarray of correlation values, same length as df
    """
    result = np.zeros(len(df))

    for rec in df[record_col].unique():
        rec_mask = df[record_col] == rec
        rec_idx = np.where(rec_mask)[0]
        waveforms = df.loc[rec_mask, wf_col].tolist()

        # Select template beats based on label_source
        if isinstance(label_source, str) and label_source == 'gt':
            n_mask = df.loc[rec_mask, 'clinical_label'].values == 'N'
        elif isinstance(label_source, str) and label_source == 'all':
            n_mask = np.ones(len(waveforms), dtype=bool)
        elif isinstance(label_source, str) and label_source == 'rr_filtered':
            rr_prev = df.loc[rec_mask, 'rr_prev'].values
            n_mask = select_on_time_beats(rr_prev, threshold=0.8)
        else:
            # Predicted labels array (aligned to df index)
            pred_labels = np.asarray(label_source)
            n_mask = pred_labels[rec_mask] == 'N'

        template_wf = [waveforms[i] for i in range(len(waveforms)) if n_mask[i]]
        template = _build_pwave_template(template_wf, min_beats=min_beats)

        if template is not None:
            corr = compute_pwave_template_corr(waveforms, template)
            result[rec_idx] = corr

    return result


def recompute_pw_pnorm(df, record_col, label_source):
    """Recompute per-patient P-wave normalization using a specified label source.

    Returns dict of {col_pnorm: np.ndarray} for each PW2_RAW_COL.
    """
    result = {f'{c}_pnorm': np.zeros(len(df)) for c in PW2_RAW_COLS}

    for rec in df[record_col].unique():
        rec_mask = df[record_col] == rec
        rec_idx = np.where(rec_mask)[0]

        if isinstance(label_source, str) and label_source == 'gt':
            n_mask = df.loc[rec_mask, 'clinical_label'].values == 'N'
        elif isinstance(label_source, str) and label_source == 'all':
            n_mask = np.ones(rec_mask.sum(), dtype=bool)
        elif isinstance(label_source, str) and label_source == 'rr_filtered':
            rr_prev = df.loc[rec_mask, 'rr_prev'].values
            n_mask = select_on_time_beats(rr_prev, threshold=0.8)
        else:
            pred_labels = np.asarray(label_source)
            n_mask = pred_labels[rec_mask] == 'N'

        for feat in PW2_RAW_COLS:
            vals = df.loc[rec_mask, feat].values
            ref_vals = vals[n_mask]
            ref_vals = ref_vals[~np.isnan(ref_vals)]
            if len(ref_vals) > 1:
                m, s = ref_vals.mean(), ref_vals.std()
                if s == 0:
                    s = 1.0
            else:
                m, s = 0.0, 1.0
            result[f'{feat}_pnorm'][rec_idx] = (vals - m) / s

    return result

In [ ]:
def make_modified_test(df_test_orig, label_source, df_all_for_idx=None):
    """Create a modified DS2 DataFrame with recomputed P-wave template features.

    DS1 features are untouched (GT labels for training is legitimate).
    Only DS2 template correlation and P-wave normalization are recomputed.

    Args:
        df_test_orig: original DS2 DataFrame (from get_train_test_split)
        label_source: 'gt', 'all', or predicted label array (aligned to df_test_orig)
        df_all_for_idx: if label_source is a full-length array aligned to df_all,
                        pass df_all to extract the correct DS2 subset
    """
    df_mod = df_test_orig.copy()

    # Recompute template correlations
    if '_pw_fixed_waveform' in df_mod.columns:
        df_mod['pw_template_corr'] = recompute_template_corr(
            df_mod, 'record', '_pw_fixed_waveform', label_source)
    if '_pw2_waveform' in df_mod.columns:
        df_mod['pw2_template_corr'] = recompute_template_corr(
            df_mod, 'record', '_pw2_waveform', label_source)

    # Recompute P-wave per-patient normalization
    pnorm = recompute_pw_pnorm(df_mod, 'record', label_source)
    for col, vals in pnorm.items():
        if col in df_mod.columns:
            df_mod[col] = vals

    return df_mod

## Condition A: Oracle (current approach — GT labels for all records)

In [ ]:
# Determine which feature set to use based on available columns
# Use FEATURE_COLS_48 (hybrid no-amp) if P-wave features exist, else FEATURE_COLS (34)
pw_features_available = 'pw_template_corr' in df_all.columns

if pw_features_available:
    FEAT_COLS = FEATURE_COLS_48
    print(f"Using {len(FEAT_COLS)}-feature hybrid set (FEATURE_COLS_48)")
else:
    FEAT_COLS = FEATURE_COLS
    print(f"Using {len(FEAT_COLS)}-feature base set (no P-wave features)")

print(f"P-wave template features: {[c for c in FEAT_COLS if 'pw' in c]}")

In [ ]:
# Train GB classifier (same config across all conditions — trained on DS1 with GT)
y_train = df_train['clinical_label'].values
labels_3 = ['N', 'S', 'V']

def train_gb(X_train, y_train):
    sw = compute_sample_weight('balanced', y_train)
    clf = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.1,
        min_samples_leaf=20, l2_regularization=1.0, random_state=42,
    )
    clf.fit(X_train, y_train, sample_weight=sw)
    return clf


def evaluate(y_true, y_pred, label=''):
    """Compute metrics dict for a condition."""
    mask_not232 = None
    acc = accuracy_score(y_true, y_pred)
    n_f1 = f1_score(y_true, y_pred, labels=['N'], average=None, zero_division=0)[0]
    s_f1 = f1_score(y_true, y_pred, labels=['S'], average=None, zero_division=0)[0]
    v_f1 = f1_score(y_true, y_pred, labels=['V'], average=None, zero_division=0)[0]
    macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return {
        'Condition': label,
        'Accuracy': acc, 'N F1': n_f1, 'S F1': s_f1, 'V F1': v_f1, 'Macro F1': macro,
    }


def evaluate_per_record(df_test_subset, y_pred, record_col='record'):
    """Per-record S F1 breakdown."""
    y_true = df_test_subset['clinical_label'].values
    records = df_test_subset[record_col].values
    rows = []
    for rec in sorted(df_test_subset[record_col].unique()):
        mask = records == rec
        true_s = (y_true[mask] == 'S').sum()
        pred_s = (y_pred[mask] == 'S').sum()
        if true_s == 0 and pred_s == 0:
            continue
        s_f1 = f1_score(y_true[mask], y_pred[mask], labels=['S'], average=None, zero_division=0)[0]
        rows.append({'record': rec, 'true_S': true_s, 'pred_S': pred_s, 'S_F1': s_f1})
    return pd.DataFrame(rows)

In [ ]:
# Condition A: Oracle (GT labels everywhere — current approach)
clf_a = train_gb(df_train[FEAT_COLS].values, y_train)
y_test = df_test['clinical_label'].values
y_pred_a = clf_a.predict(df_test[FEAT_COLS].values)

metrics_a = evaluate(y_test, y_pred_a, label='A: Oracle (GT)')
print("Condition A: Oracle")
for k, v in metrics_a.items():
    if k != 'Condition':
        print(f"  {k}: {v:.4f}")

## Condition B: Deployment-realistic (first-pass classifier for DS2 templates)

1. Train GB on DS1 **without** P-wave template features (34 base features)
2. Predict DS2 labels with this first-pass model
3. Use predicted-N beats to build DS2 P-wave templates
4. Recompute `pw_template_corr` on DS2 with these realistic templates
5. Train final GB on DS1 **with** P-wave features (using GT templates — legitimate)
6. Evaluate on DS2 with deployment-realistic templates

In [ ]:
# Step 1: First-pass classifier (no P-wave template features)
# Use the 34-feature base set
clf_firstpass = train_gb(df_train[FEATURE_COLS].values, y_train)
y_pred_firstpass = clf_firstpass.predict(df_test[FEATURE_COLS].values)

# How good is the first-pass at identifying N beats?
from sklearn.metrics import precision_recall_fscore_support
p, r, f, _ = precision_recall_fscore_support(y_test, y_pred_firstpass, labels=['N'], zero_division=0)
print(f"First-pass N-class: precision={p[0]:.3f}, recall={r[0]:.3f}, F1={f[0]:.3f}")
print(f"First-pass macro F1: {f1_score(y_test, y_pred_firstpass, average='macro', zero_division=0):.3f}")

# How many N beats correctly identified vs contamination?
n_true_n = (y_test == 'N').sum()
n_pred_n = (y_pred_firstpass == 'N').sum()
n_correct_n = ((y_test == 'N') & (y_pred_firstpass == 'N')).sum()
n_contam_s = ((y_test == 'S') & (y_pred_firstpass == 'N')).sum()
n_contam_v = ((y_test == 'V') & (y_pred_firstpass == 'N')).sum()
print(f"\nPredicted-N pool: {n_pred_n} beats")
print(f"  Truly N: {n_correct_n} ({n_correct_n/n_pred_n*100:.1f}%)")
print(f"  Actually S (contamination): {n_contam_s} ({n_contam_s/n_pred_n*100:.1f}%)")
print(f"  Actually V (contamination): {n_contam_v} ({n_contam_v/n_pred_n*100:.1f}%)")

In [ ]:
# Per-record: how clean are the predicted-N templates?
print(f"{'Record':<8} {'True N':>7} {'Pred N':>7} {'Contam S':>9} {'Contam V':>9} {'Purity':>8}")
print("-" * 52)
for rec in sorted(df_test['record'].unique()):
    mask = df_test['record'].values == rec
    yt = y_test[mask]
    yp = y_pred_firstpass[mask]
    pred_n_mask = yp == 'N'
    n_pred = pred_n_mask.sum()
    if n_pred == 0:
        continue
    truly_n = ((yt == 'N') & pred_n_mask).sum()
    contam_s = ((yt == 'S') & pred_n_mask).sum()
    contam_v = ((yt == 'V') & pred_n_mask).sum()
    purity = truly_n / n_pred
    marker = " <<<" if contam_s > 5 or contam_v > 5 else ""
    print(f"{rec:<8} {(yt=='N').sum():>7} {n_pred:>7} {contam_s:>9} {contam_v:>9} {purity:>7.1%}{marker}")

In [ ]:
# Step 2-4: Recompute DS2 template features using first-pass predictions
df_test_b = make_modified_test(df_test, y_pred_firstpass)

# Step 5-6: Same GB model trained on DS1 (with GT templates), evaluate on modified DS2
# clf_a was trained on DS1 with GT P-wave features — reuse it
y_pred_b = clf_a.predict(df_test_b[FEAT_COLS].values)

metrics_b = evaluate(y_test, y_pred_b, label='B: Deployment (first-pass)')
print("Condition B: Deployment-realistic")
for k, v in metrics_b.items():
    if k != 'Condition':
        print(f"  {k}: {v:.4f}")

## Condition C: Unsupervised (all-beat mean template, no labels)

In [ ]:
# Recompute DS2 template features using ALL beats (no label selection)
df_test_c = make_modified_test(df_test, 'all')

y_pred_c = clf_a.predict(df_test_c[FEAT_COLS].values)

metrics_c = evaluate(y_test, y_pred_c, label='C: Unsupervised (all-beat)')
print("Condition C: Unsupervised")
for k, v in metrics_c.items():
    if k != 'Condition':
        print(f"  {k}: {v:.4f}")

## Condition D: RR-filtered (on-time beats only, no labels)

Exclude premature beats using RR timing alone: beats with `rr_prev < 0.8 * median(rr_prev)`
are likely ectopic (S or V). The remaining on-time beats form the template.

This is physiologically motivated — no labels required, just the observation that
premature beats have short preceding RR intervals.

In [ ]:
# Per-record: how clean are RR-filtered template pools?
print(f"{'Record':<8} {'Total':>7} {'On-time':>8} {'Excluded':>9} {'True S excl':>12} {'True V excl':>12}")
print("-" * 60)
for rec in sorted(df_test['record'].unique()):
    mask = df_test['record'].values == rec
    rr_prev = df_test.loc[mask, 'rr_prev'].values
    yt = y_test[mask]
    on_time = select_on_time_beats(rr_prev, threshold=0.8)
    excluded = ~on_time
    n_excl = excluded.sum()
    # How many of the excluded beats are actually S or V?
    s_excl = ((yt == 'S') & excluded).sum()
    v_excl = ((yt == 'V') & excluded).sum()
    # How many S/V beats remain in the on-time pool (contamination)?
    s_remain = ((yt == 'S') & on_time).sum()
    v_remain = ((yt == 'V') & on_time).sum()
    total_s = (yt == 'S').sum()
    total_v = (yt == 'V').sum()
    print(f"{rec:<8} {mask.sum():>7} {on_time.sum():>8} {n_excl:>9} "
          f"{s_excl:>5}/{total_s:<5} {v_excl:>5}/{total_v:<5}")

# Overall template pool purity
all_rr = df_test['rr_prev'].values
all_on_time = select_on_time_beats(all_rr, threshold=0.8)
# Per-record filtering (matching how recompute_template_corr works)
per_rec_on_time = np.zeros(len(df_test), dtype=bool)
for rec in df_test['record'].unique():
    mask = df_test['record'].values == rec
    per_rec_on_time[mask] = select_on_time_beats(df_test.loc[mask, 'rr_prev'].values, threshold=0.8)

pool_total = per_rec_on_time.sum()
pool_n = ((y_test == 'N') & per_rec_on_time).sum()
pool_s = ((y_test == 'S') & per_rec_on_time).sum()
pool_v = ((y_test == 'V') & per_rec_on_time).sum()
print(f"\nRR-filtered template pool: {pool_total} beats")
print(f"  Truly N: {pool_n} ({100*pool_n/pool_total:.1f}%)")
print(f"  S contamination: {pool_s} ({100*pool_s/pool_total:.1f}%)")
print(f"  V contamination: {pool_v} ({100*pool_v/pool_total:.1f}%)")

In [ ]:
# Condition D: Recompute DS2 template features using RR-filtered beats
df_test_d = make_modified_test(df_test, 'rr_filtered')

y_pred_d = clf_a.predict(df_test_d[FEAT_COLS].values)

metrics_d = evaluate(y_test, y_pred_d, label='D: RR-filtered (no labels)')
print("Condition D: RR-filtered")
for k, v in metrics_d.items():
    if k != 'Condition':
        print(f"  {k}: {v:.4f}")

## Baseline: No P-wave features at all (34 base features)

In [ ]:
# Baseline: 34-feature model (no P-wave features)
y_pred_base = clf_firstpass.predict(df_test[FEATURE_COLS].values)

metrics_base = evaluate(y_test, y_pred_base, label='Baseline: No P-wave (34)')
print("Baseline: No P-wave features")
for k, v in metrics_base.items():
    if k != 'Condition':
        print(f"  {k}: {v:.4f}")

## Summary comparison

In [ ]:
# Aggregate results
df_results = pd.DataFrame([metrics_base, metrics_a, metrics_b, metrics_c, metrics_d])
df_results = df_results.set_index('Condition')

print("=" * 85)
print("ALL DS2 RECORDS")
print("=" * 85)
print(df_results.to_string(float_format='{:.4f}'.format))

# Delta from baseline
print("\n--- Delta vs No P-wave baseline ---")
for idx in df_results.index:
    if idx == metrics_base['Condition']:
        continue
    delta_s = df_results.loc[idx, 'S F1'] - df_results.loc[metrics_base['Condition'], 'S F1']
    delta_m = df_results.loc[idx, 'Macro F1'] - df_results.loc[metrics_base['Condition'], 'Macro F1']
    print(f"  {idx}: S F1 {delta_s:+.4f}, Macro F1 {delta_m:+.4f}")

In [ ]:
# Same analysis excluding record 232
mask_not232 = df_test['record'].values != '232'

results_excl = []
for name, y_pred in [
    (metrics_base['Condition'], y_pred_base),
    (metrics_a['Condition'], y_pred_a),
    (metrics_b['Condition'], y_pred_b),
    (metrics_c['Condition'], y_pred_c),
    (metrics_d['Condition'], y_pred_d),
]:
    results_excl.append(evaluate(y_test[mask_not232], y_pred[mask_not232], label=name))

df_results_excl = pd.DataFrame(results_excl).set_index('Condition')

print("=" * 85)
print("DS2 EXCLUDING RECORD 232")
print("=" * 85)
print(df_results_excl.to_string(float_format='{:.4f}'.format))

print("\n--- Delta vs No P-wave baseline (excl. 232) ---")
for idx in df_results_excl.index:
    if idx == metrics_base['Condition']:
        continue
    delta_s = df_results_excl.loc[idx, 'S F1'] - df_results_excl.loc[metrics_base['Condition'], 'S F1']
    delta_m = df_results_excl.loc[idx, 'Macro F1'] - df_results_excl.loc[metrics_base['Condition'], 'Macro F1']
    print(f"  {idx}: S F1 {delta_s:+.4f}, Macro F1 {delta_m:+.4f}")

## Cohen's d: `pw_template_corr` (N vs S) under each condition

In [ ]:
def cohens_d(a, b):
    """Compute Cohen's d (pooled std)."""
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(a) - np.mean(b)) / pooled_std


for feat in ['pw_template_corr', 'pw2_template_corr']:
    if feat not in df_test.columns:
        continue

    print(f"\n=== {feat} — Cohen's d (N vs S) on DS2 ===")
    for name, df_cond in [
        ('A: Oracle (GT)', df_test),
        ('B: Deployment', df_test_b),
        ('C: Unsupervised', df_test_c),
        ('D: RR-filtered', df_test_d),
    ]:
        n_vals = df_cond.loc[df_cond['clinical_label'] == 'N', feat].dropna().values
        s_vals = df_cond.loc[df_cond['clinical_label'] == 'S', feat].dropna().values
        d = cohens_d(n_vals, s_vals)
        print(f"  {name}: d={d:.3f}  (N mean={n_vals.mean():.3f}, S mean={s_vals.mean():.3f})")

    # Excluding 232
    print(f"\n  --- Excluding record 232 ---")
    for name, df_cond in [
        ('A: Oracle (GT)', df_test),
        ('B: Deployment', df_test_b),
        ('C: Unsupervised', df_test_c),
        ('D: RR-filtered', df_test_d),
    ]:
        mask = df_cond['record'] != '232'
        n_vals = df_cond.loc[mask & (df_cond['clinical_label'] == 'N'), feat].dropna().values
        s_vals = df_cond.loc[mask & (df_cond['clinical_label'] == 'S'), feat].dropna().values
        d = cohens_d(n_vals, s_vals)
        print(f"  {name}: d={d:.3f}  (N mean={n_vals.mean():.3f}, S mean={s_vals.mean():.3f})")

## Per-record S F1 comparison

In [ ]:
# Per-record S F1 for all conditions
pr_base = evaluate_per_record(df_test, y_pred_base).rename(columns={'S_F1': 'Base_S_F1'})
pr_a = evaluate_per_record(df_test, y_pred_a).rename(columns={'S_F1': 'A_S_F1'})
pr_b = evaluate_per_record(df_test, y_pred_b).rename(columns={'S_F1': 'B_S_F1'})
pr_c = evaluate_per_record(df_test, y_pred_c).rename(columns={'S_F1': 'C_S_F1'})
pr_d = evaluate_per_record(df_test, y_pred_d).rename(columns={'S_F1': 'D_S_F1'})

pr = pr_base[['record', 'true_S', 'Base_S_F1']].merge(
    pr_a[['record', 'A_S_F1']], on='record', how='outer').merge(
    pr_b[['record', 'B_S_F1']], on='record', how='outer').merge(
    pr_c[['record', 'C_S_F1']], on='record', how='outer').merge(
    pr_d[['record', 'D_S_F1']], on='record', how='outer')

pr = pr.fillna(0).sort_values('record')
pr['A_vs_D'] = pr['A_S_F1'] - pr['D_S_F1']

print("Per-record S F1 comparison:")
print(pr.to_string(index=False, float_format='{:.3f}'.format))

## Leave-one-out check for N beats

With ~1000+ N beats per record, excluding one beat from the template mean should
have negligible impact. Verify this empirically.

In [ ]:
# LOO analysis: compare correlation of N beats with vs without self-inclusion
if '_pw_fixed_waveform' in df_test.columns:
    loo_diffs = []

    for rec in sorted(df_test['record'].unique()):
        rec_mask = df_test['record'] == rec
        n_mask = rec_mask & (df_test['clinical_label'] == 'N')

        n_waveforms_raw = df_test.loc[n_mask, '_pw_fixed_waveform'].tolist()
        # Filter to valid waveforms
        n_waveforms = []
        n_indices = []
        for i, wf in enumerate(n_waveforms_raw):
            if wf is not None and hasattr(wf, '__len__') and len(wf) >= 5:
                if len(wf) != PW_RESAMPLE_LEN:
                    wf = sig_resample(wf, PW_RESAMPLE_LEN)
                n_waveforms.append(wf)
                n_indices.append(i)

        if len(n_waveforms) < 20:
            continue

        n_wf_arr = np.array(n_waveforms)
        template_all = np.mean(n_wf_arr, axis=0)  # includes self
        n_total = len(n_wf_arr)

        # For each N beat: compute correlation with template_all vs LOO template
        for j in range(min(200, n_total)):  # cap at 200 per record for speed
            wf = n_wf_arr[j]
            # LOO template: (sum - this_beat) / (n-1)
            template_loo = (template_all * n_total - wf) / (n_total - 1)

            wf_c = wf - np.mean(wf)
            if np.std(wf_c) == 0:
                continue

            t_all_c = template_all - np.mean(template_all)
            t_loo_c = template_loo - np.mean(template_loo)

            if np.std(t_all_c) > 0 and np.std(t_loo_c) > 0:
                corr_all = np.corrcoef(t_all_c, wf_c)[0, 1]
                corr_loo = np.corrcoef(t_loo_c, wf_c)[0, 1]
                loo_diffs.append({
                    'record': rec,
                    'corr_with_self': corr_all,
                    'corr_loo': corr_loo,
                    'diff': corr_all - corr_loo,
                    'n_beats': n_total,
                })

    df_loo = pd.DataFrame(loo_diffs)
    print("=== Leave-one-out impact on N-beat template correlation ===")
    print(f"Mean absolute diff: {df_loo['diff'].abs().mean():.6f}")
    print(f"Max absolute diff:  {df_loo['diff'].abs().max():.6f}")
    print(f"Mean self-included corr:  {df_loo['corr_with_self'].mean():.4f}")
    print(f"Mean LOO corr:            {df_loo['corr_loo'].mean():.4f}")
    print(f"\nPer-record summary:")
    print(df_loo.groupby('record').agg(
        n_beats=('n_beats', 'first'),
        mean_diff=('diff', 'mean'),
        max_diff=('diff', lambda x: x.abs().max()),
    ).to_string(float_format='{:.6f}'.format))
else:
    print("Waveform columns not available — skipping LOO analysis")

## Conclusions

### Label leakage in P-wave features
1. **Leakage magnitude**: Oracle (A) inflates S F1 by +0.332 over deployment-realistic (B). More than half the apparent P-wave benefit was label information leaking through template construction.
2. **Corrected 48-feature GB**: S F1 0.630 → 0.298, Macro F1 0.845 → 0.732. The previously reported numbers were inflated.
3. **P-wave features still help modestly**: All deployment-realistic approaches (B, C, D) improve S F1 by ~+0.06-0.07 over the 34-feature baseline. Real but small.
4. **RR-filtered template (D)**: Comparable to classifier-based (B) and unsupervised (C). Simpler, no labels needed, but doesn't outperform.
5. **LOO self-inclusion**: Negligible (mean diff 0.0006). Not a meaningful source of bias.

### Impact on model rankings
- **Hybrid CNN + Tabular (macro F1 0.761, S F1 0.438)**: Unaffected — uses 34 tabular features + raw waveform, no P-wave template features. Remains the best model by a wider margin than previously apparent.
- **GB no-amp 48-feat**: Corrected from macro F1 0.845 → 0.732. Still better than the 34-feature GB (0.702) but the gap is modest (+0.030), not dramatic.
- **MLP (macro F1 0.685, S F1 0.389)**: Also unaffected — uses same 34 features. Second-best S F1.

### Implication for deployment
At deployment, P-wave template features provide ~+0.07 S F1 improvement regardless of how templates are constructed (first-pass classifier, all-beat mean, or RR-filtered). The Hybrid CNN approach sidesteps this entirely by learning morphology directly from the raw waveform.

## 48-Feature GB Model: Corrected Performance

The 48-feature "GB no-amp" model (`FEATURE_COLS_48`) was previously reported as the best
GB model. It includes 14 P-wave features that depend on per-patient template/normalization
construction. When those templates are built with ground-truth N labels (oracle), DS2
performance is inflated.

This section documents the corrected numbers using deployment-realistic template construction
(Condition B: first-pass classifier) as the honest baseline.

**Affected features (14 of 48):**
- `pw_template_corr`, `pw2_template_corr` — template correlation (built from GT N-beats)
- `pw2_*_pnorm` (9 features) — z-scored against GT N-beat stats
- `pw_pca_0` through `pw_pca_4` — PCA fit on DS1 N-beats (minor: DS1 label use is legitimate,
  but PCA subspace is shaped by label selection)

**Unaffected features (34 of 48):**
- All RR timing, QRS morphology, per-patient QRS norms, robust RR, QRS PCA

In [ ]:
# --- 48-Feature GB: Oracle vs Deployment-Realistic ---
print("=" * 90)
print("48-FEATURE GB MODEL (FEATURE_COLS_48): ORACLE vs CORRECTED PERFORMANCE")
print("=" * 90)

# All conditions already used FEAT_COLS = FEATURE_COLS_48, so results are directly applicable
print(f"\nFeature set: FEATURE_COLS_48 ({len(FEAT_COLS)} features)")
print(f"P-wave features using label info: {[c for c in FEAT_COLS if 'pw' in c]}")
print(f"Count: {len([c for c in FEAT_COLS if 'pw' in c])} of {len(FEAT_COLS)} features affected\n")

# --- All DS2 ---
print("-" * 90)
print("ALL DS2 RECORDS")
print("-" * 90)
print(f"{'Condition':<40} {'Acc':>8} {'N F1':>8} {'S F1':>8} {'V F1':>8} {'Macro F1':>10}")
print("-" * 90)
for m in [metrics_base, metrics_a, metrics_b, metrics_c, metrics_d]:
    name = m['Condition']
    print(f"{name:<40} {m['Accuracy']:>8.4f} {m['N F1']:>8.4f} {m['S F1']:>8.4f} "
          f"{m['V F1']:>8.4f} {m['Macro F1']:>10.4f}")

# Highlight the key comparison
print(f"\n>>> Previously reported (oracle):     S F1 = {metrics_a['S F1']:.4f}, Macro F1 = {metrics_a['Macro F1']:.4f}")
print(f">>> Corrected (deployment-realistic): S F1 = {metrics_b['S F1']:.4f}, Macro F1 = {metrics_b['Macro F1']:.4f}")
print(f">>> Inflation from leakage:           S F1 = {metrics_a['S F1'] - metrics_b['S F1']:+.4f}, "
      f"Macro F1 = {metrics_a['Macro F1'] - metrics_b['Macro F1']:+.4f}")

# --- Excl. 232 ---
print(f"\n{'-' * 90}")
print("DS2 EXCLUDING RECORD 232")
print("-" * 90)
mask_not232 = df_test['record'].values != '232'
print(f"{'Condition':<40} {'Acc':>8} {'N F1':>8} {'S F1':>8} {'V F1':>8} {'Macro F1':>10}")
print("-" * 90)
for name, y_pred in [
    (metrics_base['Condition'], y_pred_base),
    (metrics_a['Condition'], y_pred_a),
    (metrics_b['Condition'], y_pred_b),
    (metrics_c['Condition'], y_pred_c),
    (metrics_d['Condition'], y_pred_d),
]:
    m = evaluate(y_test[mask_not232], y_pred[mask_not232], label=name)
    print(f"{name:<40} {m['Accuracy']:>8.4f} {m['N F1']:>8.4f} {m['S F1']:>8.4f} "
          f"{m['V F1']:>8.4f} {m['Macro F1']:>10.4f}")

In [ ]:
# --- Corrected All-Models Comparison ---
# The 48-feature results previously reported used oracle templates (Condition A).
# This table replaces those with Condition B (deployment-realistic) as the honest number.

print("=" * 90)
print("CORRECTED ALL-MODELS COMPARISON (DS2, inter-patient split)")
print("=" * 90)
print(f"\n{'Model':<45} {'Acc':>7} {'N F1':>7} {'S F1':>7} {'V F1':>7} {'Macro':>7} {'Leak?':>6}")
print("-" * 90)

models = [
    ("RF (28 features)",                     0.954, 0.975, 0.037, 0.927, 0.646, "No"),
    ("GB (28 features)",                     0.940, 0.967, 0.225, 0.880, 0.691, "No"),
    ("GB (34 features)",                     metrics_base['Accuracy'], metrics_base['N F1'],
     metrics_base['S F1'], metrics_base['V F1'], metrics_base['Macro F1'], "No"),
    ("GB no-amp (48) — ORACLE (was reported)", metrics_a['Accuracy'], metrics_a['N F1'],
     metrics_a['S F1'], metrics_a['V F1'], metrics_a['Macro F1'], "YES"),
    ("GB no-amp (48) — CORRECTED",           metrics_b['Accuracy'], metrics_b['N F1'],
     metrics_b['S F1'], metrics_b['V F1'], metrics_b['Macro F1'], "No"),
    ("MLP (34 features)",                    0.876, 0.929, 0.389, 0.737, 0.685, "No"),
    ("Hybrid CNN + Tabular (34 + waveform)", 0.917, 0.954, 0.438, 0.892, 0.761, "No"),
]

for name, acc, nf1, sf1, vf1, mf1, leak in models:
    print(f"{name:<45} {acc:>7.3f} {nf1:>7.3f} {sf1:>7.3f} {vf1:>7.3f} {mf1:>7.3f} {leak:>6}")

print("-" * 90)
print("\nKey corrections:")
print(f"  GB no-amp (48): S F1 {metrics_a['S F1']:.3f} -> {metrics_b['S F1']:.3f} "
      f"({metrics_b['S F1'] - metrics_a['S F1']:+.3f}), "
      f"Macro F1 {metrics_a['Macro F1']:.3f} -> {metrics_b['Macro F1']:.3f} "
      f"({metrics_b['Macro F1'] - metrics_a['Macro F1']:+.3f})")
print(f"\n  Hybrid CNN + Tabular remains best overall (macro F1 0.761, S F1 0.438)")
print(f"  Hybrid CNN uses 34 tabular features (no P-wave) + raw waveform — UNAFFECTED by leakage")
print(f"  Its S F1 advantage over corrected GB: {0.438 - metrics_b['S F1']:.3f} "
      f"(previously appeared as {0.438 - metrics_a['S F1']:+.3f})")

## Export Model Artifacts

Save trained GB classifier and supporting artifacts for the dashboard app (`dash_app.py`).
This replaces the standalone `export_models.py` script — single source of truth for training + export.

In [ ]:
import os, joblib, wfdb
from ecg_monitor.pipeline import (
    FEATURE_COLS_48, DATA_DIR, LABEL_MAP,
    bandpass_filter, compute_patient_baseline, compute_session_metrics,
)

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Train final GB classifier on full DS1 with 48 features
clf_export = train_gb(df_train[FEAT_COLS].values, y_train)

# Compute baseline session metrics per record (using predicted labels)
baselines = {}
for rid in df_all['record'].unique():
    df_rec = df_all[df_all['record'] == rid].copy()
    pred = clf_export.predict(df_rec[FEAT_COLS].values)
    df_rec['predicted_label'] = pred
    baselines[rid] = compute_session_metrics(df_rec, label_col='predicted_label')

# Compute frozen per-patient baselines (GT labels for template building)
frozen_baselines = {}
for rid in sorted(df_all['record'].unique()):
    record = wfdb.rdrecord(f'{DATA_DIR}/{rid}')
    ann = wfdb.rdann(f'{DATA_DIR}/{rid}', 'atr')
    filtered = bandpass_filter(record.p_signal[:, 0], record.fs)
    gt_labels = [LABEL_MAP.get(s, 'N') for s in ann.symbol]
    df_rec = df_all[df_all['record'] == rid]
    frozen_baselines[rid] = compute_patient_baseline(
        df_rec, filtered, record.fs, ann.sample, gt_labels)

# Build record info summary
record_info = {}
for rid in df_all['record'].unique():
    df_rec = df_all[df_all['record'] == rid]
    record_info[rid] = {
        'n_total': len(df_rec),
        'n_N': int((df_rec['clinical_label'] == 'N').sum()),
        'n_S': int((df_rec['clinical_label'] == 'S').sum()),
        'n_V': int((df_rec['clinical_label'] == 'V').sum()),
    }

# Save all artifacts
joblib.dump(clf_export, f'{MODELS_DIR}/gb_48feat.joblib')
joblib.dump(pca, f'{MODELS_DIR}/qrs_pca.joblib')
joblib.dump(pw_pca, f'{MODELS_DIR}/pw_pca.joblib')
joblib.dump(frozen_baselines, f'{MODELS_DIR}/frozen_baselines.joblib')
joblib.dump(baselines, f'{MODELS_DIR}/baselines.joblib')
joblib.dump(record_info, f'{MODELS_DIR}/record_info.joblib')

print('Artifacts saved to models/')
for name in ['gb_48feat.joblib', 'qrs_pca.joblib', 'pw_pca.joblib',
             'frozen_baselines.joblib', 'baselines.joblib', 'record_info.joblib']:
    size = os.path.getsize(f'{MODELS_DIR}/{name}')
    print(f'  {name:<30s} {size / 1024:.1f} KB')
print(f'\n{len(record_info)} records, {sum(r["n_total"] for r in record_info.values())} total beats')
print(f'Classifier: HistGradientBoosting, {len(FEAT_COLS)} features')